In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

import os

def _exists(p):
    return p and os.path.exists(p)

def resolve_ocr_dir() -> str:
    env_p = os.environ.get("OCR_DIR")
    if _exists(env_p):
        return env_p

    candidates = [
        "/content/drive/MyDrive/OCR/",
        "/content/drive/My Drive/OCR/",
    ]
    for p in candidates:
        if _exists(p):
            return p

    share_root = "/content/drive/Shareddrives"
    if _exists(share_root):
        for team in os.listdir(share_root):
            cand = os.path.join(share_root, team, "OCR")
            if _exists(cand):
                return cand

    root = "/content/drive"
    max_depth = 3
    for cur, dirs, _files in os.walk(root):
        depth = cur.count(os.sep) - root.count(os.sep)
        if depth > max_depth:
            dirs[:] = []
            continue
        if os.path.basename(cur) == "OCR":
            return cur

    raise FileNotFoundError(
        "OCR folder not found.\n"
        "Make sure you added a *shortcut* of the shared 'OCR' folder to 'My Drive', "
        "or set OCR_DIR to the exact path."

    )

OCR_DIR = resolve_ocr_dir()
print("Using OCR data at:", OCR_DIR)

if not os.path.exists("/content/OCR"):
    os.symlink(OCR_DIR, "/content/OCR")
print("Stable path:", "/content/OCR")

Mounted at /content/drive
Using OCR data at: /content/drive/MyDrive/OCR/
Stable path: /content/OCR


In [ ]:
# !pip install datasets==3.3.2

In [ ]:
# !pip -q install -U "transformers>=4.44" accelerate

In [ ]:
# !pip -q install sentence_transformers --upgrade

In [ ]:
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True,max_split_size_mb:64"
os.environ["WANDB_DISABLED"] = "true"
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")

'expandable_segments:True,max_split_size_mb:64'

### Taks Batch clean


In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-

"""
Task-batch OCR-robust fine-tuning (clean, plug-and-play)

- Exactly 1 batch per task per cycle, in the order you asked.
- Pools auto-adjust when a dataset path is empty or missing.
- Unique run names include data signature, enabled tasks, steps, bs, seed, timestamp.
- Prompt prefixes added for GTE/E5 models ("query:", "summary:", "document:") when enabled.
- LUX pairs are deduped and mirrored (a,b) & (b,a) to strengthen symmetry.
"""

import os, re, json, math, random, gc, shutil
from datetime import datetime
from types import SimpleNamespace
from typing import List, Tuple, Optional, Dict

import numpy as np
import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader
from sentence_transformers import SentenceTransformer, InputExample, losses

# --------------------------
# Repro & small utilities
# --------------------------
def set_random_seed(seed: int):
    random.seed(seed); np.random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)

def parse_seeds(spec) -> List[int]:
    if spec is None: return [42]
    if isinstance(spec, int): return [spec]
    if isinstance(spec, list): return [int(x) for x in spec]
    s = str(spec).replace(' ',''); parts = s.split(',')
    out = []
    for p in parts:
        if '-' in p:
            a,b = p.split('-',1); out.extend(range(int(a), int(b)+1))
        elif p != '':
            out.append(int(p))
    return sorted(set(out))

import hashlib

def slugify(text: str, maxlen: int = 50) -> str:
    base = re.sub(r'[^A-Za-z0-9._-]+','-', str(text)).strip('-')
    if len(base) <= maxlen:
        return base
    h = hashlib.sha1(text.encode('utf-8')).hexdigest()[:8]
    return base[:maxlen-9] + '-' + h

def short_tag_from_path(path: str, maxlen: int = 30) -> str:
    base = os.path.basename(path)
    return slugify(base, maxlen=maxlen)

# --------------------------
# IO helpers (comment-out friendly)
# --------------------------
def optional_read_csv(path: Optional[str]) -> Optional[pd.DataFrame]:
    if not path or not str(path).strip():
        print("[info] CSV path empty -> skipping"); return None
    p = str(path)
    if not os.path.exists(p):
        print(f"[warn] CSV not found at {p} -> skipping"); return None
    try:
        df = pd.read_csv(p)
        print(f"[load] {p} -> {len(df)} rows"); return df
    except Exception as e:
        print(f"[warn] Failed to read {p}: {e} -> skipping"); return None

def optional_load_jsonl(path: Optional[str]) -> Optional[List[dict]]:
    if not path or not str(path).strip():
        print("[info] JSONL path empty -> skipping"); return None
    if not os.path.exists(path):
        print(f"[warn] JSONL not found at {path} -> skipping"); return None
    dataset = []
    try:
        with open(path, "r", encoding="utf-8") as f:
            for line in f: dataset.append(json.loads(line))
        print(f"[load] {path} -> {len(dataset)} records"); return dataset
    except Exception as e:
        print(f"[warn] Failed to read {path}: {e} -> skipping"); return None

# --------------------------
# LUX helpers
# --------------------------
_slug_re = re.compile(r'[^A-Za-z0-9\s]+')
def clean_text_from_punctuation(text: str) -> str:
    return _slug_re.sub(' ', str(text)).strip()

def extract_parallel_sentences(lines: Optional[List[dict]], src_col: str, tgt_col: str, min_chars: int = 5) -> List[Tuple[str,str]]:
    if not lines: return []
    out = []
    for data in lines:
        translations = data.get("translation", [])
        for sp in translations:
            if not isinstance(sp, dict): continue
            s = sp.get(src_col); t = sp.get(tgt_col)
            if isinstance(s, list) or isinstance(t, list): continue
            if s and t:
                s_clean = clean_text_from_punctuation(s); t_clean = clean_text_from_punctuation(t)
                if len(s_clean) >= min_chars and len(t_clean) >= min_chars:
                    out.append((s.strip(), t.strip()))
    return out

def sample_pairs(pool: List[Tuple[str,str]], k: Optional[int], seed: int = 42) -> List[Tuple[str,str]]:
    if not pool: return []
    rng = random.Random(seed)
    if k is None or k < 0 or k >= len(pool): return pool[:]
    return rng.sample(pool, k)

def dedup_pairs(pairs: List[Tuple[str,str]]) -> List[Tuple[str,str]]:
    seen, out = set(), []
    for a,b in pairs:
        if (a,b) not in seen: seen.add((a,b)); out.append((a,b))
    return out

# --------------------------
# Pool builders (auto column match)
# --------------------------
def cols_exist(df: pd.DataFrame, a: str, b: str) -> bool:
    return a in df.columns and b in df.columns

def safe_pair_list_from_df(df: Optional[pd.DataFrame], left_candidates: List[str], right_candidates: List[str], name: str) -> List[Tuple[str,str]]:
    if df is None or len(df) == 0: return []
    for L in left_candidates:
        for R in right_candidates:
            if cols_exist(df, L, R):
                pairs = []
                for _, row in df[[L, R]].dropna().iterrows():
                    a = str(row[L]).strip(); b = str(row[R]).strip()
                    if a and b: pairs.append((a,b))
                if pairs:
                    print(f"[pool] {name}: using ({L} -> {R}) | {len(pairs)} pairs")
                    return pairs
    print(f"[warn] {name}: no matching column pair; skipping")
    return []

def build_pools(summary_df, doc_df_de, doc_df_fr, mono_df, lux_pairs) -> Dict[str, List[Tuple[str,str]]]:
    pools = {}
    # From query_doc_dataset_random_noise
    pools["sum2doc"]       = safe_pair_list_from_df(summary_df, ["summary","summ","abstract"], ["text","document","doc"], "summary->text")
    pools["sum2sum_noise"] = safe_pair_list_from_df(summary_df, ["summary","summ","abstract"], ["summary_noise","summary_noised","summary_noise_random05","summ_noise"], "summary->summary_noise")
    pools["q2doc"]         = safe_pair_list_from_df(summary_df, ["query","q"], ["text","document","doc"], "query->text")
    pools["doc2doc"]         = safe_pair_list_from_df(summary_df, ["text","document","doc"], ["text_noised","document_noised","doc_noised"], "doc->doc")
    pools["q2sum"]         = safe_pair_list_from_df(summary_df, ["query","q"], ["summary","summ","abstract"], "query->summary")
    # From de_docs_random_noise / fr_docs_random_noise
    pools["dd_german"]     = safe_pair_list_from_df(doc_df_de, ["text","sentence"], ["text_noised","text_noise_random05","sentence_noise_random05"], "de:text->text_noised")
    pools["dd_french"]     = safe_pair_list_from_df(doc_df_fr, ["text","sentence"], ["text_noised","text_noise_random05","sentence_noise_random05"], "fr:text->text_noised")
    # From TED_data_random_noise_concat
    pools["ted_q2q"]       = safe_pair_list_from_df(mono_df, ["sentence","text","de","fr"], ["sentence_noise_random05","text_noised","de_005","fr_005"], "TED:sentence->sentence_noise")
    # From LUX (lb <-> {de, fr, en})
    if lux_pairs:
        pools["lux_q2q"] = lux_pairs[:]
        print(f"[pool] lux_q2q: {len(pools['lux_q2q'])} pairs")
    return {k:v for k,v in pools.items() if v}

# --------------------------
# Batch schedule (exactly 1 batch per available task)
# --------------------------
TASK_ORDER = [
    ("sum2doc",       1),
    ("sum2sum_noise", 1),
    ("q2doc",         1),
    ("q2sum",         1),
    ("dd_german",     2),
    ("dd_french",     2),
    ("ted_q2q",       2),
    ("lux_q2q",       1),
]

def _maybe_prefix(text: str, role: str, enable_prompts: bool, model_name: str) -> str:
    if not enable_prompts: return text
    lower = model_name.lower()
    if   "multilingual-e5" in lower or "e5-" in lower:
        if role == "query":   return f"query: {text}"
        if role == "summary": return f"summary: {text}"
        return f"document: {text}"
    return text

def sample_batch_from_pool(pool: List[Tuple[str,str]], batch_size: int, replacement: bool, rng: random.Random):
    n = len(pool)
    if n == 0: return []
    if not replacement and batch_size <= n:
        idx = rng.sample(range(n), batch_size); return [pool[i] for i in idx]
    return [pool[rng.randrange(n)] for _ in range(batch_size)]

def build_epoch_examples(
    pools: Dict[str, List[Tuple[str,str]]],
    model_name: str,
    batch_size: int,
    batches_per_epoch: int,
    replacement: bool,
    seed: int,
    enable_prompts: bool,
    task_allowlist: Optional[List[str]] = None,
    include_lux_reverse: bool = True,
) -> List[InputExample]:
    """
    Create InputExamples for exactly `batches_per_epoch` batches,
    1 batch per task in TASK_ORDER if available (cyclic schedule).
    """
    pools = {k:v for k,v in pools.items() if v}
    if task_allowlist:
        pools = {k:v for k,v in pools.items() if k in set(task_allowlist)}

    # Mirror LUX to improve symmetry
    if include_lux_reverse and "lux_q2q" in pools:
        lux = pools["lux_q2q"]
        pools["lux_q2q"] = dedup_pairs(lux + [(b, a) for (a, b) in lux])

    active = [(k,c) for (k,c) in TASK_ORDER if k in pools]
    if not active:
        raise ValueError("No active pools found after filtering.")

    ROLE_MAP = {
        "sum2doc"      : ("summary","document"),
        "sum2sum_noise": ("summary","summary"),
        "q2doc"        : ("query","document"),
        "q2sum"        : ("query","summary"),
        "doc2doc"      : ("document","document"),
        "dd_german"    : ("document","document"),
        "dd_french"    : ("document","document"),
        "ted_q2q"      : ("document","document"),
        "lux_q2q"      : ("query","document"),
    }

    steps_per_cycle = sum(c for _,c in active)
    cycles = math.ceil(batches_per_epoch / steps_per_cycle)
    rng = random.Random(seed)
    produced, target = 0, batches_per_epoch
    examples: List[InputExample] = []

    for _ in range(cycles):
        for task_key, num_batches in active:
            pool = pools[task_key]
            lrole, rrole = ROLE_MAP.get(task_key, ("document","document"))
            for _ in range(num_batches):
                if produced >= target: break
                pairs = sample_batch_from_pool(pool, batch_size, replacement, rng)
                for (a,b) in pairs:
                    ap = _maybe_prefix(a, "query" if lrole=="query" else lrole, enable_prompts, model_name)
                    bp = _maybe_prefix(b, "document" if rrole=="document" else rrole, enable_prompts, model_name)
                    examples.append(InputExample(texts=[ap, bp], label=1.0))
                produced += 1
            if produced >= target: break
        if produced >= target: break

    expected = batch_size * target
    if len(examples) != expected:
        print(f"[warn] Built {len(examples)} vs expected {expected}.")
    print(f"[info] Active tasks: {', '.join(k for k,_ in active)} | steps/cycle={steps_per_cycle} | cycles={cycles}")
    return examples

class PrebuiltExampleDataset(Dataset):
    def __init__(self, items: List[InputExample]): self.items = items
    def __len__(self): return len(self.items)
    def __getitem__(self, idx: int): return self.items[idx]

# --------------------------
# Naming helpers
# --------------------------
def task_tags(enabled_task_keys: List[str]) -> str:
    tag_map = {"sum2doc":"SUM","sum2sum_noise":"SUMN","q2doc":"QDOC","q2sum":"QSUM","dd_german":"DE","dd_french":"FR","ted_q2q":"TED","lux_q2q":"LUX"}
    return "-".join(tag_map[k] for k in enabled_task_keys)

def make_run_name(model_name: str, data_sig: str, enabled_task_keys: List[str], steps: int, bs: int, seed: int) -> str:
    base = model_name.replace("/", "_")
    tags = task_tags(enabled_task_keys)
    now = datetime.now().strftime("%Y%m%d-%H%M%S")
    name = f"{base}+{data_sig}-steps{steps}-bs{bs}-seed{seed}-{now}"
    return name[:240]

# --------------------------
# Train once (task-batch regime)
# --------------------------


def run_two_stage(seed: int, stage1_exp: dict, stage2_exp: dict,
                  stage2_lr: Optional[float] = None, stage2_epochs: Optional[int] = None):
    """
    Train Stage-1 with stage1_exp; then continue Stage-2 from the Stage-1 saved model,
    using stage2_exp (typically LUX-only). You can optionally change lr/epochs in Stage-2.
    """
    print(f"\n================= Two-Stage Training | Seed {seed} =================")

    # ---- Stage 1 ----
    print(f"\n----- Stage 1: {stage1_exp['data_sig']} -----")
    a1 = SimpleNamespace(**base_args.__dict__.copy())
    a1.seed = seed
    for k, v in stage1_exp.items():
        setattr(a1, k, v)

    out_dir_stage1 = train_task_batches_once(a1)
    print("[stage-1 done] saved at:", out_dir_stage1)

    # free up memory between stages
    del a1
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    # ---- Stage 2 (continue from Stage 1) ----
    print(f"\n----- Stage 2: {stage2_exp['data_sig']} (init from Stage-1) -----")
    a2 = SimpleNamespace(**base_args.__dict__.copy())
    a2.seed = seed

    # IMPORTANT: initialize second stage from the first stage's saved checkpoint
    a2.model_name = out_dir_stage1

    # Optionally tweak LR/epochs for the second stage
    if stage2_lr is not None:
        a2.lr = stage2_lr
    if stage2_epochs is not None:
        a2.epochs = stage2_epochs

    # Keep other base args; then apply stage-2 experiment overrides
    for k, v in stage2_exp.items():
        setattr(a2, k, v)

    # Tag the run name so it’s obvious this is continued
    # a2.data_sig = f"{stage2_exp.get('data_sig','STAGE2')}_from-{os.path.basename(out_dir_stage1)}"
    stage1_tag = short_tag_from_path(out_dir_stage1, maxlen=30)
    a2.data_sig = f"{stage2_exp.get('data_sig','STAGE2')}_from-{stage1_tag}"


    out_dir_stage2 = train_task_batches_once(a2)
    print("[stage-2 done] saved at:", out_dir_stage2)
    return out_dir_stage1, out_dir_stage2

def run_setups(
    set_ups,
    seeds=(42,),
    stage2_lr=2e-5,
    stage2_epochs=1,
    zip_after=True,
):
    saved_paths = []

    for setup in set_ups:
        model_name = setup["model_name"]
        exps = setup["experiments"]
        # normalize to tuple
        if not isinstance(exps, (list, tuple)):
            exps = (exps,)

        model_slug = slugify(model_name)
        model_outdir = os.path.join(base_args.output_dir, model_slug)

        for s in seeds:
            print(f"\n================= Model {model_name} | Seed {s} =================")

            try:
                if len(exps) == 2:
                    # TWO-STAGE: exp[0] -> exp[1]
                    stg1 = dict(exps[0])
                    stg2 = dict(exps[1])

                    # stage-1 must know which base model to start from and where to write
                    stg1["model_name"] = model_name
                    stg1["output_dir"] = model_outdir

                    # stage-2 writes to same root; model_name will be set to stage1 path inside run_two_stage
                    stg2["output_dir"] = model_outdir

                    out1, out2 = run_two_stage(
                        seed=s,
                        stage1_exp=stg1,
                        stage2_exp=stg2,
                        stage2_lr=stage2_lr,
                        stage2_epochs=stage2_epochs,
                    )
                    saved_paths.extend([out1, out2])

                    final_dir = out2  # last stage is the deliverable
                    if zip_after:
                        base_name = final_dir
                        root_dir  = os.path.dirname(final_dir) or "."
                        base_dir  = os.path.basename(final_dir)
                        zip_path  = shutil.make_archive(base_name=base_name, format="zip",
                                                        root_dir=root_dir, base_dir=base_dir)
                        print("[zip] ->", zip_path)

                elif len(exps) == 1:
                    # SINGLE-STAGE
                    exp = dict(exps[0])
                    a = SimpleNamespace(**base_args.__dict__.copy())
                    a.seed = s
                    a.model_name = model_name
                    a.output_dir = model_outdir
                    for k, v in exp.items():
                        setattr(a, k, v)

                    out_dir = train_task_batches_once(a)
                    saved_paths.append(out_dir)

                    if zip_after:
                        base_name = out_dir
                        root_dir  = os.path.dirname(out_dir) or "."
                        base_dir  = os.path.basename(out_dir)
                        zip_path  = shutil.make_archive(base_name=base_name, format="zip",
                                                        root_dir=root_dir, base_dir=base_dir)
                        print("[zip] ->", zip_path)

                else:
                    raise ValueError("Each 'experiments' must be a single dict or a 2-tuple for two-stage.")

            except torch.cuda.OutOfMemoryError as e:
                print(f"[OOM] {model_name} | seed {s}: {e}. Consider lowering batch_size or sequence length.")
                raise
            finally:
                gc.collect()
                if torch.cuda.is_available():
                    torch.cuda.empty_cache()

    print("\nSaved runs:")
    for p in saved_paths:
        print(" -", p)

def train_task_batches_once(args):
    """
    args fields:
      model_name, batch_size, batches_per_epoch, epochs, lr, seed, replacement, amp,
      max_seq_length, enable_prompts, lux_sample_k, output_dir, task_allowlist (optional),
      document_file_de, document_file_fr, summary_file, mono, lux_file_en, lux_file_de, lux_file_fr,
      data_sig (for naming)
    """
    set_random_seed(args.seed)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"[info] Device: {device}")

    # --- load datasets (commenting out -> auto-skip) ---
    doc_df_de  = optional_read_csv(getattr(args, "document_file_de", None))
    doc_df_fr  = optional_read_csv(getattr(args, "document_file_fr", None))
    summary_df = optional_read_csv(getattr(args, "summary_file", None))
    # if summary_df is not None:
    #     if 'language' in summary_df.columns:
    #         summary_df = summary_df[summary_df['language'].isin(['de','fr','es'])]
    #     else:
    #         print("[warn] summary_df has no 'language' column; skipping language filter")


    mono_df    = optional_read_csv(getattr(args, "mono", None))

    lb_de = optional_load_jsonl(getattr(args, "lux_file_de", None))
    lb_fr = optional_load_jsonl(getattr(args, "lux_file_fr", None))
    lb_en = optional_load_jsonl(getattr(args, "lux_file_en", None))

    lb_de_pairs = extract_parallel_sentences(lb_de, src_col="lb", tgt_col="de") if lb_de else []
    lb_fr_pairs = extract_parallel_sentences(lb_fr, src_col="lb", tgt_col="fr") if lb_fr else []
    lb_en_pairs = extract_parallel_sentences(lb_en, src_col="lb", tgt_col="en") if lb_en else []

    lux_pairs = []
    if any([lb_de_pairs, lb_fr_pairs, lb_en_pairs]):
        print(f"Extracted LUX pairs: de={len(lb_de_pairs)} | fr={len(lb_fr_pairs)} | en={len(lb_en_pairs)}")
        k = None if getattr(args, "lux_sample_k", -1) == -1 else getattr(args, "lux_sample_k")
        pools_ = []
        if lb_de_pairs: pools_.extend(sample_pairs(lb_de_pairs, k, seed=args.seed))
        if lb_fr_pairs: pools_.extend(sample_pairs(lb_fr_pairs, k, seed=args.seed))
        if lb_en_pairs: pools_.extend(sample_pairs(lb_en_pairs, k, seed=args.seed))
        lux_pairs = dedup_pairs(pools_)
        print(f"Lux combined pool size (deduped): {len(lux_pairs)}")

    pools = build_pools(summary_df, doc_df_de, doc_df_fr, mono_df, lux_pairs)

    # enabled task keys in fixed order
    enabled_task_keys = [k for (k, _) in TASK_ORDER if k in pools]
    task_allowlist = getattr(args, "task_allowlist", None)
    if task_allowlist:
        allow = set(task_allowlist)
        enabled_task_keys = [k for k in enabled_task_keys if k in allow]
    if not enabled_task_keys:
        raise RuntimeError("No data pools available. Enable at least one dataset or relax task_allowlist.")

    examples = build_epoch_examples(
        pools=pools,
        model_name=args.model_name,
        batch_size=args.batch_size,
        batches_per_epoch=args.batches_per_epoch,
        replacement=args.replacement,
        seed=args.seed,
        enable_prompts=False,
        task_allowlist=task_allowlist,
        include_lux_reverse=False,
    )

    train_ds = PrebuiltExampleDataset(examples)
    model = SentenceTransformer(args.model_name, trust_remote_code=True).to(device)
    # model.max_seq_length = args.max_seq_length

    train_loader = DataLoader(
        train_ds,
        batch_size=args.batch_size,
        shuffle=False,
        drop_last=True,
        collate_fn=model.smart_batching_collate,
        num_workers=4, pin_memory=True, persistent_workers=True
    )
    train_loss = losses.MultipleNegativesRankingLoss(model=model)
#     train_loss = losses.CachedMultipleNegativesRankingLoss(
#     model,
#     mini_batch_size=8,
#     scale=30.0,              # temperature = 1/scale
#     gather_across_devices=False
# )
    warmup_steps = int(0.1 * len(train_loader))
    print(f"[info] batches_per_epoch={len(train_loader)} | warmup_steps={warmup_steps} | epochs={args.epochs}")

    os.makedirs(args.output_dir, exist_ok=True)
    run_name = make_run_name(
        model_name=args.model_name,
        data_sig=args.data_sig,
        enabled_task_keys=enabled_task_keys,
        steps=args.batches_per_epoch,
        bs=args.batch_size,
        seed=args.seed
    )
    save_dir = os.path.join(args.output_dir, run_name)
    os.makedirs(save_dir, exist_ok=True)
    print(f"[info] Output dir: {save_dir}")

    model.fit(
        train_objectives=[(train_loader, train_loss)],
        epochs=args.epochs,
        warmup_steps=warmup_steps,
        optimizer_params={"lr": args.lr},
        use_amp=args.amp,
        output_path=save_dir,
        show_progress_bar=True
    )

    del train_loader, train_ds, model
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return save_dir

# --------------------------
# Defaults + staged experiments
# --------------------------

base_args = SimpleNamespace(
    # data files (set "" to disable)
    document_file_de = f"{OCR_DIR}/finetuning_data/de_docs_random_noise.csv",
    document_file_fr = f"{OCR_DIR}/finetuning_data/fr_docs_random_noise.csv",
    summary_file     = f"{OCR_DIR}/finetuning_data/query_doc_dataset_random_noise.csv",
    mono             = f"{OCR_DIR}/finetuning_data/TED_data_random_noise_concat.csv",
    lux_file_en      = f"{OCR_DIR}/finetuning_data/lb_en_training_set.jsonl",
    lux_file_de      = f"{OCR_DIR}/finetuning_data/lb_de_training_set.jsonl",
    lux_file_fr      = f"{OCR_DIR}/finetuning_data/lb_fr_training_set.jsonl",

    # training
    # model_name = "impresso-project/histlux-gte-multilingual-base",
    model_name = "Alibaba-NLP/gte-multilingual-base",
    batch_size = 8,
    batches_per_epoch = 3000,   # ≈ 2–3k steps/epoch as requested
    epochs = 1,
    lr = 2e-5,
    seed = 42,
    replacement = False,
    amp = True,
    # max_seq_length = 512,
    enable_prompts = False,      # add 'query:' / 'document:' / 'summary:' for GTE/E5
    lux_sample_k = -1,          # -1 = all pairs
    output_dir = "./trained_models",

    # naming helper
    data_sig = "BASE",
    task_allowlist = None,
)

# --- Staged experiments you asked for ---
model_names = [
    "impresso-project/histlux-gte-multilingual-base",
    "Alibaba-NLP/gte-multilingual-base",
]

# 1) TED only
exp_TED = dict(
    data_sig="TED",
    mono=base_args.mono,
    lux_file_en="",
    lux_file_de="",
    lux_file_fr="",
    summary_file="",
    document_file_de="",
    document_file_fr="",
    task_allowlist=["ted_q2q"],
)
exp_TED_Impresso = dict(
    data_sig="TED_Impresso",
    mono=base_args.mono,
    lux_file_en="",
    lux_file_de="",
    lux_file_fr="",
    summary_file="",
    document_file_de=base_args.document_file_de,
    document_file_fr=base_args.document_file_fr,
    task_allowlist=["ted_q2q","dd_german","dd_french"],
)
exp_TED_Impresso_SUMDOCS_ = dict(
    data_sig="TED_Impresso_SUMDOCS",
    mono=base_args.mono,
    lux_file_en="",
    lux_file_de="",
    lux_file_fr="",
    summary_file=base_args.summary_file,
    document_file_de=base_args.document_file_de,
    document_file_fr=base_args.document_file_fr,
    task_allowlist=["ted_q2q","dd_german","dd_french","doc2doc"],
)
exp_TED_Impresso_SUMDOCS_SUMMARY = dict(
    data_sig="TED_Impresso_SUMDOCS_SUMMARY",
    mono=base_args.mono,
    lux_file_en="",
    lux_file_de="",
    lux_file_fr="",
    summary_file=base_args.summary_file,
    document_file_de=base_args.document_file_de,
    document_file_fr=base_args.document_file_fr,
    task_allowlist=["ted_q2q","dd_german","dd_french","doc2doc","sum2sum_noise"],
)

# 2) TED + LUX
exp_LUX = dict(
    data_sig="LUX",
    mono="",
    lux_file_en=base_args.lux_file_en,
    lux_file_de=base_args.lux_file_de,
    lux_file_fr=base_args.lux_file_fr,
    summary_file="",
    document_file_de="",
    document_file_fr="",
    task_allowlist=["lux_q2q"],
)

# 3) TED + LUX + (query -> summary) ONLY from summary_df
exp_TED_DOCS = dict(
    data_sig="TED+DOCS",
    mono=base_args.mono,
    lux_file_en="",
    lux_file_de="",
    lux_file_fr="base_args.lux_file_fr",
    summary_file=base_args.summary_file,
    document_file_de=base_args.document_file_de,
    document_file_fr=base_args.document_file_fr,
    task_allowlist=["ted_q2q","dd_german","dd_french","doc2doc"],
)



exp_TED_DOCS_SUM = dict(
    data_sig="TED+DOCS+SUM",
    mono=base_args.mono,
    lux_file_en="",
    lux_file_de="",
    lux_file_fr="",
    summary_file=base_args.summary_file,
    document_file_de=base_args.document_file_de,
    document_file_fr=base_args.document_file_fr,
    task_allowlist=["ted_q2q","dd_german","dd_french","sum2sum_noise"],
)
exp_TED_DOCS_SUM_DOC = dict(
    data_sig="TED+DOCS+SUM+SUMDOC",
    mono=base_args.mono,
    lux_file_en="",
    lux_file_de="",
    lux_file_fr="",
    summary_file="",
    document_file_de="",
    document_file_fr="",
    task_allowlist=["ted_q2q","dd_german","dd_french","sum2doc","sum2sum_noise"],
)

# exp_TED__DOCS_SUM_QUERYDOC_ALL_DEFR = dict(
#     data_sig="TED+DOCS+SUM+SUMDOC",
#     mono=base_args.mono,
#     lux_file_en=base_args.lux_file_en,
#     lux_file_de=base_args.lux_file_de,
#     lux_file_fr=base_args.lux_file_fr,
#     summary_file=base_args.summary_file,
#     document_file_de=base_args.document_file_de,
#     document_file_fr=base_args.document_file_fr,
#     task_allowlist=["ted_q2q","q2doc","sum2doc","sum2sum_noise","dd_german","dd_french"],
# )

# exp_TED__DOCS_SUM_QUERYDOC_QUERYSUM_ALL_DEFR = dict(
#     data_sig="TED+DOCS+SUM+SUMDOC+QUERYSUM+QUERYDOC",
#     mono=base_args.mono,
#     lux_file_en=base_args.lux_file_en,
#     lux_file_de=base_args.lux_file_de,
#     lux_file_fr=base_args.lux_file_fr,
#     summary_file=base_args.summary_file,
#     document_file_de=base_args.document_file_de,
#     document_file_fr=base_args.document_file_fr,
#     task_allowlist=["ted_q2q","q2sum","q2doc","sum2doc","sum2sum_noise","dd_german","dd_french"],
# )

# Choose the sequence you want to run
# EXPERIMENTS = [
#     # exp_TED_DOCS_SUMALL,
#     exp_TED_DOCS_SUM_DOC_SUMALL,
#     # exp_TED__DOCS_SUM_QUERYDOC_ALL_DEFR,
#     # exp_TED__DOCS_SUM_QUERYDOC_QUERYSUM_ALL_DEFR
#     ]

set_ups = [
    # {"model_name":"impresso-project/histlux-gte-multilingual-base", "experiments": exp_TED},
    # {"model_name":"impresso-project/histlux-gte-multilingual-base", "experiments": exp_TED_Impresso},
    # {"model_name":"impresso-project/histlux-gte-multilingual-base", "experiments": exp_TED_Impresso_SUMDOCS_},
    # {"model_name":"impresso-project/histlux-gte-multilingual-base", "experiments": exp_TED_Impresso_SUMDOCS_SUMMARY},

    # {"model_name":"Alibaba-NLP/gte-multilingual-base", "experiments": (exp_TED, exp_LUX)},
    # {"model_name":"Alibaba-NLP/gte-multilingual-base", "experiments": (exp_TED_Impresso, exp_LUX)},
    # {"model_name":"Alibaba-NLP/gte-multilingual-base", "experiments": (exp_TED_Impresso_SUMDOCS_, exp_LUX)},
    # {"model_name":"Alibaba-NLP/gte-multilingual-base", "experiments": (exp_TED_Impresso_SUMDOCS_SUMMARY, exp_LUX)},

    # {"model_name":"Alibaba-NLP/gte-multilingual-base", "experiments": exp_TED},
    # {"model_name":"Alibaba-NLP/gte-multilingual-base", "experiments": exp_TED_Impresso},
    # {"model_name":"Alibaba-NLP/gte-multilingual-base", "experiments": exp_TED_Impresso_SUMDOCS_},
    {"model_name":"Alibaba-NLP/gte-multilingual-base", "experiments": exp_TED_Impresso_SUMDOCS_SUMMARY},
]


# STAGE1 = exp_TED_DOCS
# STAGE2 = exp_LUX

# --------------------------
# Run loop
# --------------------------
seeds = [100]
# seeds = [42,100, 123, 777, 999]
zip_after = True
run_setups(
    set_ups=set_ups,
    seeds=seeds,
    stage2_lr=2e-5,
    stage2_epochs=1,
    zip_after=True,
)
saved_paths = []
# for model_name in model_names:
# for s in seeds:
#     print(f"\n================= Seed {s} =================")

#     out1, out2 = run_two_stage(
#         seed=s,
#         stage1_exp=STAGE1,
#         stage2_exp=STAGE2,
#         stage2_lr=2e-5,      # optionally lower to 1e-5 if you want gentler adaptation
#         stage2_epochs=1      # bump to 2–3 if LUX is small and you want more passes
#     )
#     saved_paths.extend([out1, out2])
#     final_dir = out2
#     base_name = final_dir                          # zip will be final_dir + ".zip"
#     root_dir  = os.path.dirname(final_dir) or "."
#     base_dir  = os.path.basename(final_dir)
#     zip_path  = shutil.make_archive(base_name=base_name, format="zip",
#                                     root_dir=root_dir, base_dir=base_dir)

#     # cleanup between seeds
#     gc.collect()
#     if torch.cuda.is_available():
#         torch.cuda.empty_cache()

    # for exp in EXPERIMENTS:
    #     print(f"\n----- Experiment {exp['data_sig']} -----")
    #     a = SimpleNamespace(**base_args.__dict__.copy())
    #     a.seed = s
    #     for k,v in exp.items():
    #         setattr(a, k, v)

    #     try:
    #         out_dir = train_task_batches_once(a)
    #         saved_paths.append(out_dir)

    #         if zip_after:
    #             base_name = out_dir
    #             root_dir = os.path.dirname(out_dir) or "."
    #             base_dir = os.path.basename(out_dir)
    #             zip_path = shutil.make_archive(base_name=base_name, format="zip", root_dir=root_dir, base_dir=f"{OCR_DIR}/trained_models/")
    #             print("[zip] ->", zip_path)

            # # cleanup between runs
            # del a
            # gc.collect()
            # if torch.cuda.is_available():
            #     torch.cuda.empty_cache()

        # except torch.cuda.OutOfMemoryError as e:
        #     print(f"[OOM @ seed {s}, exp {exp['data_sig']}] Reduce bs/seq length. {e}")
        #     raise
        # except Exception as e:
        #     print(f"[Error @ seed {s}, exp {exp['data_sig']}] {e}")
        #     raise

print("\nSaved runs:")
for p in saved_paths:
    print(" -", p)



================= Model Alibaba-NLP/gte-multilingual-base | Seed 100 =================
[info] Device: cuda
[load] /content/drive/MyDrive/OCR//finetuning_data/de_docs_random_noise.csv -> 10000 rows
[load] /content/drive/MyDrive/OCR//finetuning_data/fr_docs_random_noise.csv -> 10000 rows
[load] /content/drive/MyDrive/OCR//finetuning_data/query_doc_dataset_random_noise.csv -> 10000 rows
[load] /content/drive/MyDrive/OCR//finetuning_data/TED_data_random_noise_concat.csv -> 348658 rows
[info] JSONL path empty -> skipping
[info] JSONL path empty -> skipping
[info] JSONL path empty -> skipping
[pool] summary->text: using (summary -> text) | 10000 pairs
[pool] summary->summary_noise: using (summary -> summary_noised) | 10000 pairs
[pool] query->text: using (query -> text) | 10000 pairs
[pool] doc->doc: using (text -> text_noised) | 10000 pairs
[pool] query->summary: using (query -> summary) | 10000 pairs
[pool] de:text->text_noised: using (text -> text_noised) | 10000 pairs
[pool] fr:text->te

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/55.0 [00:00<?, ?B/s]

...

[info] batches_per_epoch=3000 | warmup_steps=300 | epochs=1
[info] Output dir: ./trained_models/Alibaba-NLP-gte-multilingual-base/Alibaba-NLP_gte-multilingual-base+TED_Impresso_SUMDOCS_SUMMARY-steps3000-bs8-seed100-20251023-080541


Using the `WANDB_DISABLED` environment variable is deprecated and will be removed in v5. Use the --report_to flag to control the integrations used for logging result (for instance --report_to none).
Using the `WANDB_DISABLED` environment variable is deprecated and will be removed in v5. Use the --report_to flag to control the integrations used for logging result (for instance --report_to none).


Computing widget examples:   0%|          | 0/1 [00:00<?, ?example/s]

Step,Training Loss
500,0.000300
1000,0.000400
1500,0.000000
2000,0.000000
2500,0.000000
3000,0.000000


[zip] -> /content/trained_models/Alibaba-NLP-gte-multilingual-base/Alibaba-NLP_gte-multilingual-base+TED_Impresso_SUMDOCS_SUMMARY-steps3000-bs8-seed100-20251023-080541.zip

Saved runs:
 - ./trained_models/Alibaba-NLP-gte-multilingual-base/Alibaba-NLP_gte-multilingual-base+TED_Impresso_SUMDOCS_SUMMARY-steps3000-bs8-seed100-20251023-080541

Saved runs:


## Evaluation

In [ ]:
!pip install rank_bm25

### **CLSD**

In [ ]:
from pathlib import Path
from sentence_transformers import SentenceTransformer

# 1) Choose a local folder name (avoid slashes)
local_name = "impresso-project/histlux-gte-multilingual-base"
target_dir = Path("trained_models") / local_name
target_dir.mkdir(parents=True, exist_ok=True)



# 2) Load from HF and save locally
model = SentenceTransformer(
    local_name,
    trust_remote_code=True
)
model.save(str(target_dir))  # writes config.json, modules.json, weights, etc.

print(f"Saved to: {target_dir.resolve()}")

# 3) Later, load it from the local folder
local_model = SentenceTransformer(str(target_dir), trust_remote_code=True)
embeddings = model.encode(["hello world"], normalize_embeddings=True)
local_model = model
print(embeddings.shape)
print(local_model)


Saved to: /content/trained_models/impresso-project/histlux-gte-multilingual-base
(1, 768)
SentenceTransformer(
  (0): Transformer({'max_seq_length': 8192, 'do_lower_case': False, 'architecture': 'NewModel'})
  (1): Pooling({'word_embedding_dimension': 768, 'pooling_mode_cls_token': True, 'pooling_mode_mean_tokens': False, 'pooling_mode_max_tokens': False, 'pooling_mode_mean_sqrt_len_tokens': False, 'pooling_mode_weightedmean_tokens': False, 'pooling_mode_lasttoken': False, 'include_prompt': True})
  (2): Normalize()
)


In [ ]:
import os
import pandas as pd
from sentence_transformers import util
import torch
from tqdm import tqdm
from itertools import product

In [ ]:

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

def find_model_directories(prefix):
    # model_base_dir = os.path.join(OCR_DIR, "trained_models")
    model_base_dir = os.path.join(os.getcwd(), "trained_models")

    if not os.path.exists(model_base_dir):
        return []

    return [os.path.join(model_base_dir, d) for d in os.listdir(model_base_dir) if os.path.isdir(os.path.join(model_base_dir, d)) and d.startswith(prefix)]

def _is_st_model_dir(path: str) -> bool:
    return (
        os.path.isfile(os.path.join(path, "config.json")) and
        os.path.isfile(os.path.join(path, "modules.json"))
    )

def _resolve_model_root(path: str, max_depth: int = 3) -> str | None:
    """
    If `path` is not an ST root, descend up to `max_depth` times through
    single-child directories to find one that has config.json+modules.json.
    Handles zips that created an extra top-level folder.
    """
    if _is_st_model_dir(path):
        return path

    cur = path
    for _ in range(max_depth):
        try:
            children = [d for d in os.listdir(cur) if os.path.isdir(os.path.join(cur, d))]
        except FileNotFoundError:
            return None
        if len(children) != 1:
            return None
        nxt = os.path.join(cur, children[0])
        if _is_st_model_dir(nxt):
            return nxt
        cur = nxt
    return None

# def find_model_directories(prefix):
#     if "/" in prefix:                        # e.g., "Alibaba-NLP/gte-multilingual-base"
#         return [prefix]
#     model_base_dir = os.path.join(OCR_DIR, "trained_models")
#     if not os.path.exists(model_base_dir):
#         return []
#     out = []
#     for d in os.listdir(model_base_dir):
#         full = os.path.join(model_base_dir, d)
#         if not (os.path.isdir(full) and d.startswith(prefix)):
#             continue
#         st_root = _resolve_model_root(full)
#         if st_root:
#             out.append(st_root)
#         else:
#             try:
#                 print(f"[skip] Not an ST root: {full} -> {os.listdir(full)[:5]}")
#             except Exception:
#                 print(f"[skip] Not an ST root: {full}")
#     return sorted(out)


def encode_texts(model, texts, prefix=""):
    return model.encode([prefix + t for t in texts], convert_to_tensor=True).to(device)

def compare_languages(model, left_df, right_df, main_col, versions, prefix=""):
    main_embeddings = encode_texts(model, left_df[main_col].tolist(), prefix)
    results = pd.DataFrame()

    for version in versions:
        version_embeddings = encode_texts(model, right_df[version].tolist(), prefix)
        similarity_scores = util.cos_sim(main_embeddings, version_embeddings).diag().cpu().numpy()
        results[version] = similarity_scores

    max_scores = results.idxmax(axis=1)
    max_values = results.max(axis=1)
    is_max = results.eq(max_values, axis=0)
    max_counts = is_max.sum(axis=1)
    unique_max = max_counts == 1
    correct_version = versions[0]
    correct = (max_scores == correct_version) & unique_max

    return correct.mean() * 100

def process_and_save_results(tasks):
    for task in tasks:
        results = []  #
        task_name = task['name']
        prefixes = task['prefixes']
        levels = task['levels']
        prefix = task.get('prefix', "")
        versions_dict = task['versions_dict']
        eval_dataset = task['eval_dataset']

        print("Processing Task Name: " + task_name)

        save_name = prefixes[0] + "-".join(levels.keys()) + eval_dataset + f"-{task_name.replace(' ', '-')}-evaluations.csv"
        # save_name = "Alibaba-NLP_gte-multilingual-base-gradientAcc-taskpure-2025-09-20-quota2000" + "-".join(levels.keys()) + eval_dataset + f"-{task_name.replace(' ', '-')}-evaluations.csv"

        for model_prefix in tqdm(prefixes, desc=f"Processing Task {task_name}"):
            model_dirs = find_model_directories(model_prefix)
            if not model_dirs:
                print(f"No models found for prefix '{model_prefix}'")
                continue

            prefix_results = []

            for model_dir in model_dirs:
                try:
                    model = SentenceTransformer(model_dir, trust_remote_code=True)
                    model.to(device)
                except ValueError as e:
                    print(f"Error loading model '{model_dir}': {e}")
                    continue

                comparisons = [
                    {
                        'name': f'{left.capitalize()} to {right.capitalize()}',
                        'left_levels': [left],
                        'right_levels': [right],
                    }
                    for left, right in product(levels.keys(), repeat=2)
                ]

                for comparison in comparisons:
                    for left_level in comparison['left_levels']:
                        for right_level in comparison['right_levels']:
                            left_df = pd.read_csv(levels[left_level])
                            right_df = pd.read_csv(levels[right_level])

                            for compare_col in versions_dict:
                                versions = versions_dict[compare_col]
                                if not all(v in right_df.columns for v in versions):
                                    print(f"Missing columns for {versions} in {levels[right_level]}")
                                    continue

                                percentage = compare_languages(
                                    model, left_df, right_df, compare_col, versions, prefix
                                )

                                left_lang = 'DE' if compare_col == 'German' else 'FR'
                                right_lang = 'FR' if compare_col == 'German' else 'DE'
                                language_direction = f"{left_lang}_{left_level}->{right_lang}_{right_level}"

                                prefix_results.append({
                                    'Model': model_dir,
                                    'Comparison': comparison['name'],
                                    'Corruption Level': f"{left_level} to {right_level}",
                                    'Language Direction': language_direction,
                                    'Accuracy': percentage
                                })

            prefix_results_df = pd.DataFrame(prefix_results)
            if not prefix_results_df.empty:
                avg_results = prefix_results_df.groupby(
                    ['Comparison', 'Corruption Level', 'Language Direction']
                )['Accuracy'].mean().reset_index()
                avg_results['Model Prefix'] = model_prefix
                results.extend(avg_results.to_dict('records'))

        results_df = pd.DataFrame(results)
        results_df.to_csv(f"{OCR_DIR}/results/"+save_name, index=False)
        print(f"Results for task '{task_name}' and model prefix saved to {save_name}")



Using device: cuda


In [ ]:
import os, re

# set your real base dir (yours seem to be on Drive)
MODELS_DIR = os.path.join(OCR_DIR, "trained_models")

def dir_base_name(dname: str) -> str:
    """
    Return the canonical base name:
      - strip '-steps...' suffix if present
      - keep full name for folders without steps (e.g., '+LUX' ends)
    """
    m = re.match(r'^(.*?)-steps\d.*$', dname)
    return m.group(1) if m else dname

def list_model_dirs(base_dir=MODELS_DIR):
    return [d for d in os.listdir(base_dir)
            if os.path.isdir(os.path.join(base_dir, d))]

def find_model_directories_by_base(basename: str, base_dir=MODELS_DIR):
    """
    Match ONLY this exact base:
      - exact folder equals basename  OR
      - folder starts with basename + '-steps'
    """
    out = []
    for d in list_model_dirs(base_dir):
        if d == basename or d.startswith(basename + "-steps"):
            out.append(os.path.join(base_dir, d))
    return sorted(out)

# Define exact basenames (no '-steps...' suffix)
ALL_PREFIXES = [
    # Alibaba family (exact)
    "Alibaba-NLP_gte-multilingual-base+TED",
    "Alibaba-NLP_gte-multilingual-base+TED_Impresso",
    "Alibaba-NLP_gte-multilingual-base+TED_Impresso_SUMDOCS",
    "Alibaba-NLP_gte-multilingual-base+TED_Impresso_SUMDOCS_SUMMARY",

    # impresso-project family (exact)
    "impresso-project_histlux-gte-multilingual-base+TED",
    "impresso-project_histlux-gte-multilingual-base+TED_Impresso",
    "impresso-project_histlux-gte-multilingual-base+TED_Impresso_SUMDOCS",
    "impresso-project_histlux-gte-multilingual-base+TED_Impresso_SUMDOCS_SUMMARY",

    # trained_* (no steps; keep full names)
    "trained_Alibaba-NLP+TED_Impresso+LUX",
    "trained_Alibaba-NLP+TED_Impresso_SUMDOCS+LUX",
    "trained_Alibaba-NLP+TED_Impresso_SUMDOCS_SUMMARY",
    "trained_Alibaba-NLP-+TED+LUX",  # odd, but keep as-is

    # ---------- Hugging Face model IDs ----------
    "Alibaba-NLP/gte-multilingual-base",
    "impresso-project/histlux-gte-multilingual-base",
    "impresso-project/ocr-quality-assessor-unigram-light",
]

# (Optional) sanity check: show matches per base
for b in ALL_PREFIXES:
    print(b, "->", find_model_directories_by_base(b))


Alibaba-NLP_gte-multilingual-base+TED -> ['/content/drive/MyDrive/OCR/trained_models/Alibaba-NLP_gte-multilingual-base+TED-steps3000-bs8-seed42-20251003-180200']
Alibaba-NLP_gte-multilingual-base+TED_Impresso -> ['/content/drive/MyDrive/OCR/trained_models/Alibaba-NLP_gte-multilingual-base+TED_Impresso-steps3000-bs8-seed42-20251002-174748']
Alibaba-NLP_gte-multilingual-base+TED_Impresso_SUMDOCS -> ['/content/drive/MyDrive/OCR/trained_models/Alibaba-NLP_gte-multilingual-base+TED_Impresso_SUMDOCS-steps3000-bs8-seed42-20251002-180614']
Alibaba-NLP_gte-multilingual-base+TED_Impresso_SUMDOCS_SUMMARY -> ['/content/drive/MyDrive/OCR/trained_models/Alibaba-NLP_gte-multilingual-base+TED_Impresso_SUMDOCS_SUMMARY-steps3000-bs8-seed42-20251002-182442']
impresso-project_histlux-gte-multilingual-base+TED -> ['/content/drive/MyDrive/OCR/trained_models/impresso-project_histlux-gte-multilingual-base+TED-steps3000-bs8-seed42-20251002-112626']
impresso-project_histlux-gte-multilingual-base+TED_Impresso ->

In [ ]:
tasks = [
    {
        'name': 'Simple Noise',
        'prefixes': [
            "impresso-project_histlux-gte-multilingual-base+TED_Impresso_SUMDOCS_SUMMARY-steps3000-bs8-seed100-20251022-110331",
            "Alibaba-NLP_gte-multilingual-base+TED_Impresso_SUMDOCS_SUMMARY-steps3000-bs8-seed100-20251022-112015"
        ],
        'levels': {
            'clean': f'{OCR_DIR}/evaluation_sets/CLSD_wmt2019_adversarial_dataset.csv',
            'simple': f'{OCR_DIR}/evaluation_sets/CLSD_WMT19_MN_noise.csv'
        },
        'versions_dict': {
            'German': ['French', 'fr_adv1', 'fr_adv2', 'fr_adv3', 'fr_adv4'],
            'French': ['German', 'de_adv1', 'de_adv2', 'de_adv3', 'de_adv4']
        },
        'prefix': "",
        'eval_dataset': "CLSD-wmt19-"
    },
    {
        'name': 'Simple Noise',
        'prefixes': [
            "impresso-project_histlux-gte-multilingual-base+TED_Impresso_SUMDOCS_SUMMARY-steps3000-bs8-seed100-20251022-110331",
            "Alibaba-NLP_gte-multilingual-base+TED_Impresso_SUMDOCS_SUMMARY-steps3000-bs8-seed100-20251022-112015"
        ],
        'levels': {
            'clean': f'{OCR_DIR}/evaluation_sets/CLSD_wmt2021_adversarial_dataset.csv',
            'simple': f'{OCR_DIR}/evaluation_sets/CLSD_WMT21_MN_noise.csv'
        },
        'versions_dict': {
            'German': ['French', 'fr_adv1', 'fr_adv2', 'fr_adv3', 'fr_adv4'],
            'French': ['German', 'de_adv1', 'de_adv2', 'de_adv3', 'de_adv4']
        },
        'prefix': "",
        'eval_dataset': "CLSD-wmt21-"
    },
    {
        'name': 'Blackletter-Scanned Noise',
        'prefixes': [
            "impresso-project_histlux-gte-multilingual-base+TED_Impresso_SUMDOCS_SUMMARY-steps3000-bs8-seed100-20251022-110331",
            "Alibaba-NLP_gte-multilingual-base+TED_Impresso_SUMDOCS_SUMMARY-steps3000-bs8-seed100-20251022-112015"
        ],
        'levels': {
            'clean': f'{OCR_DIR}/evaluation_sets/CLSD_wmt2019_adversarial_dataset.csv',
            'bl-distorted': f'{OCR_DIR}/evaluation_sets/CLSD_WMT19_BLDS_noise.csv'
        },
        'versions_dict': {
            'German': ['French', 'fr_adv1', 'fr_adv2', 'fr_adv3', 'fr_adv4'],
            'French': ['German', 'de_adv1', 'de_adv2', 'de_adv3', 'de_adv4']
        },
        'prefix': "",
        'eval_dataset': "CLSD-wmt19-"
    },
    {
        'name': 'Blackletter-Scanned Noise',
        'prefixes': [
            "impresso-project_histlux-gte-multilingual-base+TED_Impresso_SUMDOCS_SUMMARY-steps3000-bs8-seed100-20251022-110331",
            "Alibaba-NLP_gte-multilingual-base+TED_Impresso_SUMDOCS_SUMMARY-steps3000-bs8-seed100-20251022-112015"
        ],
        'levels': {
            'clean': f'{OCR_DIR}/evaluation_sets/CLSD_wmt2021_adversarial_dataset.csv',
            'bl-distorted': f'{OCR_DIR}/evaluation_sets/CLSD_WMT21_BLDS_noise.csv'
        },
        'versions_dict': {
            'German': ['French', 'fr_adv1', 'fr_adv2', 'fr_adv3', 'fr_adv4'],
            'French': ['German', 'de_adv1', 'de_adv2', 'de_adv3', 'de_adv4']
        },
        'prefix': "",
        'eval_dataset': "CLSD-wmt21-"
    },
    {
        'name': 'SaltnPepper Noise',
        'prefixes': [
            "impresso-project_histlux-gte-multilingual-base+TED_Impresso_SUMDOCS_SUMMARY-steps3000-bs8-seed100-20251022-110331",
            "Alibaba-NLP_gte-multilingual-base+TED_Impresso_SUMDOCS_SUMMARY-steps3000-bs8-seed100-20251022-112015"
        ],
        'levels': {
            'clean': f'{OCR_DIR}/evaluation_sets/CLSD_wmt2019_adversarial_dataset.csv',
            'SaltnPepper': f'{OCR_DIR}/evaluation_sets/CLSD_WMT19_SNP_noise.csv'
        },
        'versions_dict': {
            'German': ['French', 'fr_adv1', 'fr_adv2', 'fr_adv3', 'fr_adv4'],
            'French': ['German', 'de_adv1', 'de_adv2', 'de_adv3', 'de_adv4']
        },
        'prefix': "",
        'eval_dataset': "CLSD-wmt19-"
    },
    {
        'name': 'SaltnPepper Noise',
        'prefixes': [
            "impresso-project_histlux-gte-multilingual-base+TED_Impresso_SUMDOCS_SUMMARY-steps3000-bs8-seed100-20251022-110331",
            "Alibaba-NLP_gte-multilingual-base+TED_Impresso_SUMDOCS_SUMMARY-steps3000-bs8-seed100-20251022-112015"
        ],
        'levels': {
            'clean': f'{OCR_DIR}/evaluation_sets/CLSD_wmt2021_adversarial_dataset.csv',
            'SaltnPepper': f'{OCR_DIR}/evaluation_sets/CLSD_WMT21_SNP_noise.csv'
        },
        'versions_dict': {
            'German': ['French', 'fr_adv1', 'fr_adv2', 'fr_adv3', 'fr_adv4'],
            'French': ['German', 'de_adv1', 'de_adv2', 'de_adv3', 'de_adv4']
        },
        'prefix': "",
        'eval_dataset': "CLSD-wmt21-"
    }
]


In [ ]:
process_and_save_results(tasks)

Processing Task Name: Simple Noise


Processing Task Simple Noise:   0%|          | 0/2 [00:00<?, ?it/s]The module name impresso_hyphen_project_histlux_hyphen_gte_hyphen_multilingual_hyphen_base+TED_Impresso_SUMDOCS_SUMMARY_hyphen_steps3000_hyphen_bs8_hyphen_seed100_hyphen_20251022_hyphen_110331 (originally impresso-project_histlux-gte-multilingual-base+TED_Impresso_SUMDOCS_SUMMARY-steps3000-bs8-seed100-20251022-110331) is not a valid Python identifier. Please rename the original module to avoid import issues.
The module name impresso_hyphen_project_histlux_hyphen_gte_hyphen_multilingual_hyphen_base+TED_Impresso_SUMDOCS_SUMMARY_hyphen_steps3000_hyphen_bs8_hyphen_seed100_hyphen_20251022_hyphen_110331 (originally impresso-project_histlux-gte-multilingual-base+TED_Impresso_SUMDOCS_SUMMARY-steps3000-bs8-seed100-20251022-110331) is not a valid Python identifier. Please rename the original module to avoid import issues.
The module name impresso_hyphen_project_histlux_hyphen_gte_hyphen_multilingual_hyphen_base+TED_Impresso_SUMDO

Results for task 'Simple Noise' and model prefix saved to impresso-project_histlux-gte-multilingual-base+TED_Impresso_SUMDOCS_SUMMARY-steps3000-bs8-seed100-20251022-110331clean-simpleCLSD-wmt19--Simple-Noise-evaluations.csv
Processing Task Name: Simple Noise


Processing Task Simple Noise:   0%|          | 0/2 [00:00<?, ?it/s]The module name impresso_hyphen_project_histlux_hyphen_gte_hyphen_multilingual_hyphen_base+TED_Impresso_SUMDOCS_SUMMARY_hyphen_steps3000_hyphen_bs8_hyphen_seed100_hyphen_20251022_hyphen_110331 (originally impresso-project_histlux-gte-multilingual-base+TED_Impresso_SUMDOCS_SUMMARY-steps3000-bs8-seed100-20251022-110331) is not a valid Python identifier. Please rename the original module to avoid import issues.
The module name impresso_hyphen_project_histlux_hyphen_gte_hyphen_multilingual_hyphen_base+TED_Impresso_SUMDOCS_SUMMARY_hyphen_steps3000_hyphen_bs8_hyphen_seed100_hyphen_20251022_hyphen_110331 (originally impresso-project_histlux-gte-multilingual-base+TED_Impresso_SUMDOCS_SUMMARY-steps3000-bs8-seed100-20251022-110331) is not a valid Python identifier. Please rename the original module to avoid import issues.
The module name impresso_hyphen_project_histlux_hyphen_gte_hyphen_multilingual_hyphen_base+TED_Impresso_SUMDO

Results for task 'Simple Noise' and model prefix saved to impresso-project_histlux-gte-multilingual-base+TED_Impresso_SUMDOCS_SUMMARY-steps3000-bs8-seed100-20251022-110331clean-simpleCLSD-wmt21--Simple-Noise-evaluations.csv
Processing Task Name: Blackletter-Scanned Noise


...

Results for task 'Blackletter-Scanned Noise' and model prefix saved to impresso-project_histlux-gte-multilingual-base+TED_Impresso_SUMDOCS_SUMMARY-steps3000-bs8-seed100-20251022-110331clean-bl-distortedCLSD-wmt21--Blackletter-Scanned-Noise-evaluations.csv
Processing Task Name: SaltnPepper Noise


Processing Task SaltnPepper Noise:   0%|          | 0/2 [00:00<?, ?it/s]The module name impresso_hyphen_project_histlux_hyphen_gte_hyphen_multilingual_hyphen_base+TED_Impresso_SUMDOCS_SUMMARY_hyphen_steps3000_hyphen_bs8_hyphen_seed100_hyphen_20251022_hyphen_110331 (originally impresso-project_histlux-gte-multilingual-base+TED_Impresso_SUMDOCS_SUMMARY-steps3000-bs8-seed100-20251022-110331) is not a valid Python identifier. Please rename the original module to avoid import issues.
The module name impresso_hyphen_project_histlux_hyphen_gte_hyphen_multilingual_hyphen_base+TED_Impresso_SUMDOCS_SUMMARY_hyphen_steps3000_hyphen_bs8_hyphen_seed100_hyphen_20251022_hyphen_110331 (originally impresso-project_histlux-gte-multilingual-base+TED_Impresso_SUMDOCS_SUMMARY-steps3000-bs8-seed100-20251022-110331) is not a valid Python identifier. Please rename the original module to avoid import issues.
The module name impresso_hyphen_project_histlux_hyphen_gte_hyphen_multilingual_hyphen_base+TED_Impresso_

Results for task 'SaltnPepper Noise' and model prefix saved to impresso-project_histlux-gte-multilingual-base+TED_Impresso_SUMDOCS_SUMMARY-steps3000-bs8-seed100-20251022-110331clean-SaltnPepperCLSD-wmt19--SaltnPepper-Noise-evaluations.csv
Processing Task Name: SaltnPepper Noise


Processing Task SaltnPepper Noise:   0%|          | 0/2 [00:00<?, ?it/s]The module name impresso_hyphen_project_histlux_hyphen_gte_hyphen_multilingual_hyphen_base+TED_Impresso_SUMDOCS_SUMMARY_hyphen_steps3000_hyphen_bs8_hyphen_seed100_hyphen_20251022_hyphen_110331 (originally impresso-project_histlux-gte-multilingual-base+TED_Impresso_SUMDOCS_SUMMARY-steps3000-bs8-seed100-20251022-110331) is not a valid Python identifier. Please rename the original module to avoid import issues.
The module name impresso_hyphen_project_histlux_hyphen_gte_hyphen_multilingual_hyphen_base+TED_Impresso_SUMDOCS_SUMMARY_hyphen_steps3000_hyphen_bs8_hyphen_seed100_hyphen_20251022_hyphen_110331 (originally impresso-project_histlux-gte-multilingual-base+TED_Impresso_SUMDOCS_SUMMARY-steps3000-bs8-seed100-20251022-110331) is not a valid Python identifier. Please rename the original module to avoid import issues.
The module name impresso_hyphen_project_histlux_hyphen_gte_hyphen_multilingual_hyphen_base+TED_Impresso_

Results for task 'SaltnPepper Noise' and model prefix saved to impresso-project_histlux-gte-multilingual-base+TED_Impresso_SUMDOCS_SUMMARY-steps3000-bs8-seed100-20251022-110331clean-SaltnPepperCLSD-wmt21--SaltnPepper-Noise-evaluations.csv


In [ ]:
tasks = [
    {
        'name': 'Simple Noise',
        'prefixes': [
            "Alibaba2Steps"
        ],
        'levels': {
            'clean': f'{OCR_DIR}/evaluation_sets/CLSD_wmt2019_adversarial_dataset.csv',
            'simple': f'{OCR_DIR}/evaluation_sets/CLSD_WMT19_MN_noise.csv'
        },
        'versions_dict': {
            'German': ['French', 'fr_adv1', 'fr_adv2', 'fr_adv3', 'fr_adv4'],
            'French': ['German', 'de_adv1', 'de_adv2', 'de_adv3', 'de_adv4']
        },
        'prefix': "",
        'eval_dataset': "CLSD-wmt19-"
    },
    {
        'name': 'Simple Noise',
        'prefixes': [
            "Alibaba2Steps"
                    ],
        'levels': {
            'clean': f'{OCR_DIR}/evaluation_sets/CLSD_wmt2021_adversarial_dataset.csv',
            'simple': f'{OCR_DIR}/evaluation_sets/CLSD_WMT21_MN_noise.csv'
        },
        'versions_dict': {
            'German': ['French', 'fr_adv1', 'fr_adv2', 'fr_adv3', 'fr_adv4'],
            'French': ['German', 'de_adv1', 'de_adv2', 'de_adv3', 'de_adv4']
        },
        'prefix': "",
        'eval_dataset': "CLSD-wmt21-"
    },
    {
        'name': 'Blackletter-Scanned Noise',
        'prefixes': [
            "Alibaba2Steps"
            ],
        'levels': {
            'clean': f'{OCR_DIR}/evaluation_sets/CLSD_wmt2019_adversarial_dataset.csv',
            'bl-distorted': f'{OCR_DIR}/evaluation_sets/CLSD_WMT19_BLDS_noise.csv'
        },
        'versions_dict': {
            'German': ['French', 'fr_adv1', 'fr_adv2', 'fr_adv3', 'fr_adv4'],
            'French': ['German', 'de_adv1', 'de_adv2', 'de_adv3', 'de_adv4']
        },
        'prefix': "",
        'eval_dataset': "CLSD-wmt19-"
    },
    {
        'name': 'Blackletter-Scanned Noise',
        'prefixes': [
            "Alibaba2Steps"
            ],
        'levels': {
            'clean': f'{OCR_DIR}/evaluation_sets/CLSD_wmt2021_adversarial_dataset.csv',
            'bl-distorted': f'{OCR_DIR}/evaluation_sets/CLSD_WMT21_BLDS_noise.csv'
        },
        'versions_dict': {
            'German': ['French', 'fr_adv1', 'fr_adv2', 'fr_adv3', 'fr_adv4'],
            'French': ['German', 'de_adv1', 'de_adv2', 'de_adv3', 'de_adv4']
        },
        'prefix': "",
        'eval_dataset': "CLSD-wmt21-"
    },
    {
        'name': 'SaltnPepper Noise',
        'prefixes': [
            "Alibaba2Steps"
            ],
        'levels': {
            'clean': f'{OCR_DIR}/evaluation_sets/CLSD_wmt2019_adversarial_dataset.csv',
            'SaltnPepper': f'{OCR_DIR}/evaluation_sets/CLSD_WMT19_SNP_noise.csv'
        },
        'versions_dict': {
            'German': ['French', 'fr_adv1', 'fr_adv2', 'fr_adv3', 'fr_adv4'],
            'French': ['German', 'de_adv1', 'de_adv2', 'de_adv3', 'de_adv4']
        },
        'prefix': "",
        'eval_dataset': "CLSD-wmt19-"
    },
    {
        'name': 'SaltnPepper Noise',
        'prefixes': [
            "Alibaba2Steps"
            ],
        'levels': {
            'clean': f'{OCR_DIR}/evaluation_sets/CLSD_wmt2021_adversarial_dataset.csv',
            'SaltnPepper': f'{OCR_DIR}/evaluation_sets/CLSD_WMT21_SNP_noise.csv'
        },
        'versions_dict': {
            'German': ['French', 'fr_adv1', 'fr_adv2', 'fr_adv3', 'fr_adv4'],
            'French': ['German', 'de_adv1', 'de_adv2', 'de_adv3', 'de_adv4']
        },
        'prefix': "",
        'eval_dataset': "CLSD-wmt21-"
    }
]

process_and_save_results(tasks)

Processing Task Name: Simple Noise


Processing Task Simple Noise: 100%|██████████| 1/1 [01:00<00:00, 60.18s/it]


Results for task 'Simple Noise' and model prefix saved to Alibaba2Stepsclean-simpleCLSD-wmt19--Simple-Noise-evaluations.csv
Processing Task Name: Simple Noise


Processing Task Simple Noise: 100%|██████████| 1/1 [00:41<00:00, 41.54s/it]


Results for task 'Simple Noise' and model prefix saved to Alibaba2Stepsclean-simpleCLSD-wmt21--Simple-Noise-evaluations.csv
Processing Task Name: Blackletter-Scanned Noise


...

Results for task 'Blackletter-Scanned Noise' and model prefix saved to Alibaba2Stepsclean-bl-distortedCLSD-wmt21--Blackletter-Scanned-Noise-evaluations.csv
Processing Task Name: SaltnPepper Noise


Processing Task SaltnPepper Noise: 100%|██████████| 1/1 [01:02<00:00, 62.21s/it]


Results for task 'SaltnPepper Noise' and model prefix saved to Alibaba2Stepsclean-SaltnPepperCLSD-wmt19--SaltnPepper-Noise-evaluations.csv
Processing Task Name: SaltnPepper Noise


Processing Task SaltnPepper Noise: 100%|██████████| 1/1 [00:42<00:00, 42.48s/it]

Results for task 'SaltnPepper Noise' and model prefix saved to Alibaba2Stepsclean-SaltnPepperCLSD-wmt21--SaltnPepper-Noise-evaluations.csv


In [ ]:
# # From the OCR directory, extract into trained_models/
# %cd "/content/drive/MyDrive/OCR"
# !unzip -o "Alibaba-NLP_gte-multilingual-base-mixed8-seed42.zip" -d "trained_models/"


In [ ]:
"""
FIXED: CLSD Extended Evaluation Script
Properly handles trust_remote_code for all models

Based on your successful run, this version ensures all models load correctly.
"""

import os
import torch
import pandas as pd
from sentence_transformers import SentenceTransformer, util
from itertools import product
from tqdm import tqdm
from datetime import datetime

# Configuration
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")


def encode_texts(model, texts, prefix=""):
    """Encode texts with optional prefix."""
    return model.encode([prefix + t for t in texts], convert_to_tensor=True).to(device)


def compare_languages(model, left_df, right_df, main_col, versions, prefix=""):
    """Original comparison for symmetric noise configurations."""
    main_embeddings = encode_texts(model, left_df[main_col].tolist(), prefix)
    results = pd.DataFrame()

    for version in versions:
        version_embeddings = encode_texts(model, right_df[version].tolist(), prefix)
        similarity_scores = util.cos_sim(main_embeddings, version_embeddings).diag().cpu().numpy()
        results[version] = similarity_scores

    max_scores = results.idxmax(axis=1)
    max_values = results.max(axis=1)
    is_max = results.eq(max_values, axis=0)
    max_counts = is_max.sum(axis=1)
    unique_max = max_counts == 1
    correct_version = versions[0]
    correct = (max_scores == correct_version) & unique_max

    return correct.mean() * 100


def compare_languages_mixed_noise(
    model,
    source_df,
    target_df,
    distractors_df,
    main_col,
    versions,
    prefix=""
):
    """Mixed-noise comparison for asymmetric configurations."""
    # Encode source queries
    source_embeddings = encode_texts(model, source_df[main_col].tolist(), prefix)

    results = pd.DataFrame()

    # Encode correct target (first version)
    correct_version = versions[0]
    target_embeddings = encode_texts(model, target_df[correct_version].tolist(), prefix)
    correct_similarities = util.cos_sim(source_embeddings, target_embeddings).diag().cpu().numpy()
    results[correct_version] = correct_similarities

    # Encode distractors (remaining versions)
    for version in versions[1:]:
        distractor_embeddings = encode_texts(model, distractors_df[version].tolist(), prefix)
        similarity_scores = util.cos_sim(source_embeddings, distractor_embeddings).diag().cpu().numpy()
        results[version] = similarity_scores

    # Calculate accuracy
    max_scores = results.idxmax(axis=1)
    max_values = results.max(axis=1)
    is_max = results.eq(max_values, axis=0)
    max_counts = is_max.sum(axis=1)
    unique_max = max_counts == 1
    correct = (max_scores == correct_version) & unique_max

    return correct.mean() * 100


def evaluate_single_model(model_name, tasks, include_mixed_configs=True):
    """Evaluate a single model on all tasks."""
    print(f"\n{'='*80}")
    print(f"MODEL: {model_name}")
    print(f"{'='*80}")

    # Load model with proper trust_remote_code handling
    try:
        # ALWAYS use trust_remote_code=True for all your models
        model = SentenceTransformer(model_name, trust_remote_code=True)
        model.to(device)
        print(f"✓ Loaded with trust_remote_code=True")
        print(f"✓ Moved to device: {device}")
    except Exception as e:
        print(f"✗ Failed to load model: {e}")
        return None

    all_results = []

    for task_idx, task in enumerate(tasks, 1):
        task_name = task['name']
        levels = task['levels']
        prefix = task.get('prefix', "")
        versions_dict = task['versions_dict']
        eval_dataset = task['eval_dataset']

        print(f"\n[Task {task_idx}/{len(tasks)}] {task_name} - {eval_dataset}")
        print("-" * 80)

        # ===============================================================
        # PART 1: ORIGINAL COMPARISONS
        # ===============================================================
        comparisons = [
            {'name': f'{left.capitalize()} to {right.capitalize()}',
             'left_level': left, 'right_level': right}
            for left, right in product(levels.keys(), repeat=2)
        ]

        print(f"Original configurations: {len(comparisons)} comparisons")

        for comparison in tqdm(comparisons, desc="Original", leave=False):
            left_level = comparison['left_level']
            right_level = comparison['right_level']

            left_df = pd.read_csv(levels[left_level])
            right_df = pd.read_csv(levels[right_level])

            for compare_col in versions_dict:
                versions = versions_dict[compare_col]

                if not all(v in right_df.columns for v in versions):
                    print(f"  ⚠️  Missing columns in {right_level}")
                    continue

                percentage = compare_languages(
                    model, left_df, right_df, compare_col, versions, prefix
                )

                left_lang = 'DE' if compare_col == 'German' else 'FR'
                right_lang = 'FR' if compare_col == 'German' else 'DE'

                all_results.append({
                    'Model': model_name,
                    'Task': task_name,
                    'Dataset': eval_dataset,
                    'Comparison': comparison['name'],
                    'Source_Level': left_level,
                    'Target_Level': right_level,
                    'Distractor_Level': right_level,
                    'Config_Type': 'Original',
                    'Language': f"{left_lang}->{right_lang}",
                    'Accuracy': percentage
                })

        # ===============================================================
        # PART 2: NEW MIXED-NOISE CONFIGURATIONS
        # ===============================================================
        if include_mixed_configs:
            # Identify clean and noisy levels
            clean_level = None
            noisy_levels = []

            for level_name in levels.keys():
                if 'clean' in level_name.lower():
                    clean_level = level_name
                else:
                    noisy_levels.append(level_name)

            if clean_level is None:
                print("  ⚠️  No 'clean' level found, skipping mixed configs")
            else:
                print(f"Mixed configurations: {len(noisy_levels) * 2} comparisons")

                for noisy_level in tqdm(noisy_levels, desc="Mixed", leave=False):
                    # Config 1: Clean → Clean (Noisy Distractors)
                    source_df = pd.read_csv(levels[clean_level])
                    target_df = pd.read_csv(levels[clean_level])
                    distractors_df = pd.read_csv(levels[noisy_level])

                    for compare_col in versions_dict:
                        versions = versions_dict[compare_col]

                        percentage = compare_languages_mixed_noise(
                            model, source_df, target_df, distractors_df,
                            compare_col, versions, prefix
                        )

                        left_lang = 'DE' if compare_col == 'German' else 'FR'
                        right_lang = 'FR' if compare_col == 'German' else 'DE'

                        all_results.append({
                            'Model': model_name,
                            'Task': task_name,
                            'Dataset': eval_dataset,
                            'Comparison': f'Clean->Clean (Dist:{noisy_level})',
                            'Source_Level': clean_level,
                            'Target_Level': clean_level,
                            'Distractor_Level': noisy_level,
                            'Config_Type': 'CleanTarget-NoisyDist',
                            'Language': f"{left_lang}->{right_lang}",
                            'Accuracy': percentage
                        })

                    # Config 2: Noisy → Clean (Noisy Distractors)
                    source_df = pd.read_csv(levels[noisy_level])
                    target_df = pd.read_csv(levels[clean_level])
                    distractors_df = pd.read_csv(levels[noisy_level])

                    for compare_col in versions_dict:
                        versions = versions_dict[compare_col]

                        percentage = compare_languages_mixed_noise(
                            model, source_df, target_df, distractors_df,
                            compare_col, versions, prefix
                        )

                        left_lang = 'DE' if compare_col == 'German' else 'FR'
                        right_lang = 'FR' if compare_col == 'German' else 'DE'

                        all_results.append({
                            'Model': model_name,
                            'Task': task_name,
                            'Dataset': eval_dataset,
                            'Comparison': f'{noisy_level}->Clean (Dist:{noisy_level})',
                            'Source_Level': noisy_level,
                            'Target_Level': clean_level,
                            'Distractor_Level': noisy_level,
                            'Config_Type': 'NoisySource-CleanTarget',
                            'Language': f"{left_lang}->{right_lang}",
                            'Accuracy': percentage
                        })

        print(f"✓ Completed {len([r for r in all_results if r['Task'] == task_name])} evaluations")

    return pd.DataFrame(all_results)


def main():
    """Main evaluation function."""

    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

    # FIXED: All models now use trust_remote_code=True
    models = [
        'impresso-project/halloween_workshop_ocr_robust_with_lux_preview',
        'impresso-project/halloween_workshop_ocr_robust_preview',  # NOW WILL WORK
        'impresso-project/histlux-gte-multilingual-base',
        'Alibaba-NLP/gte-multilingual-base',  # NOW WILL WORK
    ]

    # Define your evaluation tasks
    tasks = [
        # WMT 2019 tasks
        {
            'name': 'Simple Noise',
            'levels': {
                'clean': f'{OCR_DIR}/evaluation_sets/CLSD_wmt2019_adversarial_dataset.csv',
                'simple': f'{OCR_DIR}/evaluation_sets/CLSD_WMT19_MN_noise.csv'
            },
            'versions_dict': {
                'German': ['French', 'fr_adv1', 'fr_adv2', 'fr_adv3', 'fr_adv4'],
                'French': ['German', 'de_adv1', 'de_adv2', 'de_adv3', 'de_adv4']
            },
            'prefix': "",
            'eval_dataset': "CLSD-wmt19"
        },
        {
            'name': 'Blackletter-Scanned Noise',
            'levels': {
                'clean': f'{OCR_DIR}/evaluation_sets/CLSD_wmt2019_adversarial_dataset.csv',
                'bl-distorted': f'{OCR_DIR}/evaluation_sets/CLSD_WMT19_BLDS_noise.csv'
            },
            'versions_dict': {
                'German': ['French', 'fr_adv1', 'fr_adv2', 'fr_adv3', 'fr_adv4'],
                'French': ['German', 'de_adv1', 'de_adv2', 'de_adv3', 'de_adv4']
            },
            'prefix': "",
            'eval_dataset': "CLSD-wmt19"
        },
        {
            'name': 'SaltnPepper Noise',
            'levels': {
                'clean': f'{OCR_DIR}/evaluation_sets/CLSD_wmt2019_adversarial_dataset.csv',
                'SaltnPepper': f'{OCR_DIR}/evaluation_sets/CLSD_WMT19_SNP_noise.csv'
            },
            'versions_dict': {
                'German': ['French', 'fr_adv1', 'fr_adv2', 'fr_adv3', 'fr_adv4'],
                'French': ['German', 'de_adv1', 'de_adv2', 'de_adv3', 'de_adv4']
            },
            'prefix': "",
            'eval_dataset': "CLSD-wmt19"
        },
        # WMT 2021 tasks
        {
            'name': 'Simple Noise',
            'levels': {
                'clean': f'{OCR_DIR}/evaluation_sets/CLSD_wmt2021_adversarial_dataset.csv',
                'simple': f'{OCR_DIR}/evaluation_sets/CLSD_WMT21_MN_noise.csv'
            },
            'versions_dict': {
                'German': ['French', 'fr_adv1', 'fr_adv2', 'fr_adv3', 'fr_adv4'],
                'French': ['German', 'de_adv1', 'de_adv2', 'de_adv3', 'de_adv4']
            },
            'prefix': "",
            'eval_dataset': "CLSD-wmt21"
        },
        {
            'name': 'Blackletter-Scanned Noise',
            'levels': {
                'clean': f'{OCR_DIR}/evaluation_sets/CLSD_wmt2021_adversarial_dataset.csv',
                'bl-distorted': f'{OCR_DIR}/evaluation_sets/CLSD_WMT21_BLDS_noise.csv'
            },
            'versions_dict': {
                'German': ['French', 'fr_adv1', 'fr_adv2', 'fr_adv3', 'fr_adv4'],
                'French': ['German', 'de_adv1', 'de_adv2', 'de_adv3', 'de_adv4']
            },
            'prefix': "",
            'eval_dataset': "CLSD-wmt21"
        },
        {
            'name': 'SaltnPepper Noise',
            'levels': {
                'clean': f'{OCR_DIR}/evaluation_sets/CLSD_wmt2021_adversarial_dataset.csv',
                'SaltnPepper': f'{OCR_DIR}/evaluation_sets/CLSD_WMT21_SNP_noise.csv'
            },
            'versions_dict': {
                'German': ['French', 'fr_adv1', 'fr_adv2', 'fr_adv3', 'fr_adv4'],
                'French': ['German', 'de_adv1', 'de_adv2', 'de_adv3', 'de_adv4']
            },
            'prefix': "",
            'eval_dataset': "CLSD-wmt21"
        }
    ]

    print("\n" + "="*80)
    print("CLSD EXTENDED EVALUATION - OCR-ROBUST MODELS (FIXED)")
    print("="*80)
    print(f"Timestamp: {timestamp}")
    print(f"Models to evaluate: {len(models)}")
    print(f"Tasks per model: {len(tasks)}")
    print(f"Device: {device}")
    print(f"✓ All models will use trust_remote_code=True")
    print("="*80)

    # Create output directory
    output_dir = os.path.join(OCR_DIR, 'results')
    os.makedirs(output_dir, exist_ok=True)

    # Evaluate all models
    all_model_results = []
    successful_models = []
    failed_models = []

    for model_idx, model_name in enumerate(models, 1):
        print(f"\n\n{'#'*80}")
        print(f"# MODEL {model_idx}/{len(models)}: {model_name}")
        print(f"{'#'*80}")

        try:
            model_results = evaluate_single_model(
                model_name,
                tasks,
                include_mixed_configs=True
            )

            if model_results is not None and not model_results.empty:
                all_model_results.append(model_results)
                successful_models.append(model_name)

                # Save individual model results
                model_short_name = model_name.replace('/', '_').replace('-', '_')
                individual_path = os.path.join(
                    output_dir,
                    f'{model_short_name}_extended_clsd_{timestamp}.csv'
                )
                model_results.to_csv(individual_path, index=False)
                print(f"\n✓ Saved: {individual_path}")
                print(f"  Rows: {len(model_results)}")
            else:
                failed_models.append(model_name)
                print(f"\n✗ No results for {model_name}")

        except Exception as e:
            failed_models.append(model_name)
            print(f"\n✗ Error evaluating {model_name}: {e}")

    # Combine and save all results
    if all_model_results:
        print("\n" + "="*80)
        print("SAVING COMBINED RESULTS")
        print("="*80)

        combined_df = pd.concat(all_model_results, ignore_index=True)

        # Save combined results
        combined_path = os.path.join(
            output_dir,
            f'all_models_extended_clsd_{timestamp}.csv'
        )
        combined_df.to_csv(combined_path, index=False)
        print(f"\n✓ Combined results: {combined_path}")
        print(f"  Total rows: {len(combined_df)}")
        print(f"  Models: {combined_df['Model'].nunique()}")
        print(f"  Tasks: {combined_df['Task'].nunique()}")

        # Generate summary
        print("\n" + "="*80)
        print("SUMMARY BY MODEL AND CONFIGURATION TYPE")
        print("="*80)

        summary = combined_df.groupby(['Model', 'Config_Type']).agg({
            'Accuracy': ['mean', 'std', 'min', 'max', 'count']
        }).round(2)

        print(summary.to_string())

        # Save summary
        summary_path = os.path.join(
            output_dir,
            f'summary_extended_clsd_{timestamp}.csv'
        )
        summary.to_csv(summary_path)
        print(f"\n✓ Summary saved: {summary_path}")

        # Print per-model averages for each config type
        print("\n" + "="*80)
        print("AVERAGE ACCURACY BY MODEL AND CONFIG TYPE")
        print("="*80)

        pivot = combined_df.pivot_table(
            values='Accuracy',
            index='Model',
            columns='Config_Type',
            aggfunc='mean'
        ).round(2)

        # Reorder columns for readability
        col_order = ['Original', 'CleanTarget-NoisyDist', 'NoisySource-CleanTarget']
        pivot = pivot[[col for col in col_order if col in pivot.columns]]

        print(pivot.to_string())

        # Save pivot table
        pivot_path = os.path.join(
            output_dir,
            f'pivot_by_config_type_{timestamp}.csv'
        )
        pivot.to_csv(pivot_path)
        print(f"\n✓ Pivot table saved: {pivot_path}")

    # Final summary
    print("\n" + "="*80)
    print("EVALUATION COMPLETE!")
    print("="*80)
    print(f"Successful models: {len(successful_models)}/{len(models)}")
    for model in successful_models:
        print(f"  ✓ {model}")
    if failed_models:
        print(f"\nFailed models: {len(failed_models)}/{len(models)}")
        for model in failed_models:
            print(f"  ✗ {model}")
    print(f"\nResults saved to: {output_dir}")
    print("="*80)


if __name__ == "__main__":
    main()

Using device: cuda

CLSD EXTENDED EVALUATION - OCR-ROBUST MODELS (FIXED)
Timestamp: 20251031_172146
Models to evaluate: 4
Tasks per model: 6
Device: cuda
✓ All models will use trust_remote_code=True


################################################################################
# MODEL 1/4: impresso-project/halloween_workshop_ocr_robust_with_lux_preview
################################################################################

MODEL: impresso-project/halloween_workshop_ocr_robust_with_lux_preview


modeling.py: 0.00B [00:00, ?B/s]

A new version of the following files was downloaded from https://huggingface.co/impresso-project/halloween_workshop_ocr_robust_with_lux_preview:
- modeling.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.


model.safetensors:   0%|          | 0.00/1.22G [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

...

Mixed configurations: 2 comparisons


✓ Completed 12 evaluations

[Task 3/6] SaltnPepper Noise - CLSD-wmt19
--------------------------------------------------------------------------------
Original configurations: 4 comparisons


Mixed configurations: 2 comparisons


✓ Completed 12 evaluations

[Task 4/6] Simple Noise - CLSD-wmt21
--------------------------------------------------------------------------------
Original configurations: 4 comparisons


Original:  75%|███████▌  | 3/4 [00:28<00:09,  9.62s/it]

In [ ]:
"""
CLSD Extended Evaluation with Cross-Noise Experiments

Adds new configurations:
1. Source BL/SD, Target SNP, Distractors SNP (cross-noise query)
2. Source BL/SD, Target BL/SD, Distractors SNP (mixed distractors)
3. Source SNP, Target BL/SD, Distractors BL/SD (reverse cross-noise)
4. Source SNP, Target SNP, Distractors BL/SD (reverse mixed distractors)
"""

import os
import torch
import pandas as pd
from sentence_transformers import SentenceTransformer, util
from itertools import product
from tqdm import tqdm
from datetime import datetime

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")


def encode_texts(model, texts, prefix=""):
    """Encode texts with optional prefix."""
    return model.encode([prefix + t for t in texts], convert_to_tensor=True).to(device)


def compare_languages_cross_noise(
    model,
    source_df,
    target_df,
    distractors_df,
    main_col,
    versions,
    prefix=""
):
    """
    Compare with separate DataFrames for source, target, and distractors.
    Allows different noise types for each component.
    """
    # Encode source queries
    source_embeddings = encode_texts(model, source_df[main_col].tolist(), prefix)

    results = pd.DataFrame()

    # Encode correct target (first version)
    correct_version = versions[0]
    target_embeddings = encode_texts(model, target_df[correct_version].tolist(), prefix)
    correct_similarities = util.cos_sim(source_embeddings, target_embeddings).diag().cpu().numpy()
    results[correct_version] = correct_similarities

    # Encode distractors (remaining versions)
    for version in versions[1:]:
        distractor_embeddings = encode_texts(model, distractors_df[version].tolist(), prefix)
        similarity_scores = util.cos_sim(source_embeddings, distractor_embeddings).diag().cpu().numpy()
        results[version] = similarity_scores

    # Calculate accuracy
    max_scores = results.idxmax(axis=1)
    max_values = results.max(axis=1)
    is_max = results.eq(max_values, axis=0)
    max_counts = is_max.sum(axis=1)
    unique_max = max_counts == 1
    correct = (max_scores == correct_version) & unique_max

    return correct.mean() * 100


def evaluate_cross_noise_experiments(model_name, tasks):
    """
    Evaluate cross-noise experiments where source and target have different noise types.
    """
    print(f"\n{'='*80}")
    print(f"MODEL: {model_name}")
    print(f"{'='*80}")

    # Load model
    try:
        model = SentenceTransformer(model_name, trust_remote_code=True)
        model.to(device)
        print(f"✓ Loaded with trust_remote_code=True")
        print(f"✓ Moved to device: {device}")
    except Exception as e:
        print(f"✗ Failed to load model: {e}")
        return None

    all_results = []

    for task_idx, task in enumerate(tasks, 1):
        task_name = task['name']
        levels = task['levels']
        prefix = task.get('prefix', "")
        versions_dict = task['versions_dict']
        eval_dataset = task['eval_dataset']

        print(f"\n[Task {task_idx}/{len(tasks)}] {task_name} - {eval_dataset}")
        print("-" * 80)

        # Identify noise levels
        clean_level = None
        bl_level = None
        snp_level = None

        for level_name in levels.keys():
            if 'clean' in level_name.lower():
                clean_level = level_name
            elif 'bl-distorted' in level_name.lower() or 'bl' in level_name.lower():
                bl_level = level_name
            elif 'saltnpepper' in level_name.lower() or 'snp' in level_name.lower():
                snp_level = level_name

        if not all([clean_level, bl_level, snp_level]):
            print(f"  ⚠️  Missing required noise levels (need: clean, BL, SnP)")
            continue

        print(f"  Found levels: clean={clean_level}, BL={bl_level}, SnP={snp_level}")

        # ===============================================================
        # CROSS-NOISE EXPERIMENTS
        # ===============================================================

        experiments = [
            {
                'name': 'BL/SD→SnP (Both noisy)',
                'source_level': bl_level,
                'target_level': snp_level,
                'distractor_level': snp_level,
                'description': 'BL query to SnP target with SnP distractors'
            },
            {
                'name': 'BL/SD→BL/SD (SnP distractors)',
                'source_level': bl_level,
                'target_level': bl_level,
                'distractor_level': snp_level,
                'description': 'BL query to BL target with SnP distractors'
            },
            {
                'name': 'SnP→BL/SD (Both noisy)',
                'source_level': snp_level,
                'target_level': bl_level,
                'distractor_level': bl_level,
                'description': 'SnP query to BL target with BL distractors'
            },
            {
                'name': 'SnP→SnP (BL distractors)',
                'source_level': snp_level,
                'target_level': snp_level,
                'distractor_level': bl_level,
                'description': 'SnP query to SnP target with BL distractors'
            },
        ]

        print(f"\nCross-noise experiments: {len(experiments)} configurations × 2 languages = {len(experiments) * 2} evaluations")

        for experiment in tqdm(experiments, desc="Cross-noise", leave=False):
            source_df = pd.read_csv(levels[experiment['source_level']])
            target_df = pd.read_csv(levels[experiment['target_level']])
            distractors_df = pd.read_csv(levels[experiment['distractor_level']])

            for compare_col in versions_dict:
                versions = versions_dict[compare_col]

                percentage = compare_cross_noise(
                    model, source_df, target_df, distractors_df,
                    compare_col, versions, prefix
                )

                left_lang = 'DE' if compare_col == 'German' else 'FR'
                right_lang = 'FR' if compare_col == 'German' else 'DE'

                all_results.append({
                    'Model': model_name,
                    'Task': task_name,
                    'Dataset': eval_dataset,
                    'Experiment': experiment['name'],
                    'Description': experiment['description'],
                    'Source_Level': experiment['source_level'],
                    'Target_Level': experiment['target_level'],
                    'Distractor_Level': experiment['distractor_level'],
                    'Config_Type': 'CrossNoise',
                    'Language': f"{left_lang}->{right_lang}",
                    'Accuracy': percentage
                })

        print(f"✓ Completed {len([r for r in all_results if r['Task'] == task_name and r['Dataset'] == eval_dataset])} cross-noise evaluations")

    return pd.DataFrame(all_results)


def main():
    """Main evaluation function."""

    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

    # All models
    models = [
        'impresso-project/halloween_workshop_ocr_robust_with_lux_preview',
        'impresso-project/halloween_workshop_ocr_robust_preview',
        'impresso-project/histlux-gte-multilingual-base',
        'Alibaba-NLP/gte-multilingual-base',
    ]

    # Tasks with both BL/SD and SnP noise
    tasks = [
        # WMT 2019
        {
            'name': 'Cross-Noise BL/SD-SnP',
            'levels': {
                'clean': f'{OCR_DIR}/evaluation_sets/CLSD_wmt2019_adversarial_dataset.csv',
                'bl-distorted': f'{OCR_DIR}/evaluation_sets/CLSD_WMT19_BLDS_noise.csv',
                'SaltnPepper': f'{OCR_DIR}/evaluation_sets/CLSD_WMT19_SNP_noise.csv'
            },
            'versions_dict': {
                'German': ['French', 'fr_adv1', 'fr_adv2', 'fr_adv3', 'fr_adv4'],
                'French': ['German', 'de_adv1', 'de_adv2', 'de_adv3', 'de_adv4']
            },
            'prefix': "",
            'eval_dataset': "CLSD-wmt19"
        },
        # WMT 2021
        {
            'name': 'Cross-Noise BL/SD-SnP',
            'levels': {
                'clean': f'{OCR_DIR}/evaluation_sets/CLSD_wmt2021_adversarial_dataset.csv',
                'bl-distorted': f'{OCR_DIR}/evaluation_sets/CLSD_WMT21_BLDS_noise.csv',
                'SaltnPepper': f'{OCR_DIR}/evaluation_sets/CLSD_WMT21_SNP_noise.csv'
            },
            'versions_dict': {
                'German': ['French', 'fr_adv1', 'fr_adv2', 'fr_adv3', 'fr_adv4'],
                'French': ['German', 'de_adv1', 'de_adv2', 'de_adv3', 'de_adv4']
            },
            'prefix': "",
            'eval_dataset': "CLSD-wmt21"
        }
    ]

    print("\n" + "="*80)
    print("CLSD CROSS-NOISE EVALUATION")
    print("="*80)
    print(f"Timestamp: {timestamp}")
    print(f"Models to evaluate: {len(models)}")
    print(f"Tasks per model: {len(tasks)}")
    print(f"Device: {device}")
    print("\nNew Experiments:")
    print("  1. BL/SD→SnP (Both noisy) - BL query, SnP target+distractors")
    print("  2. BL/SD→BL/SD (SnP distractors) - BL query+target, SnP distractors")
    print("  3. SnP→BL/SD (Both noisy) - SnP query, BL target+distractors")
    print("  4. SnP→SnP (BL distractors) - SnP query+target, BL distractors")
    print("="*80)

    # Create output directory
    output_dir = os.path.join(OCR_DIR, 'results')
    os.makedirs(output_dir, exist_ok=True)

    # Evaluate all models
    all_model_results = []
    successful_models = []
    failed_models = []

    for model_idx, model_name in enumerate(models, 1):
        print(f"\n\n{'#'*80}")
        print(f"# MODEL {model_idx}/{len(models)}: {model_name}")
        print(f"{'#'*80}")

        try:
            model_results = evaluate_cross_noise_experiments(model_name, tasks)

            if model_results is not None and not model_results.empty:
                all_model_results.append(model_results)
                successful_models.append(model_name)

                # Save individual model results
                model_short_name = model_name.replace('/', '_').replace('-', '_')
                individual_path = os.path.join(
                    output_dir,
                    f'{model_short_name}_cross_noise_{timestamp}.csv'
                )
                model_results.to_csv(individual_path, index=False)
                print(f"\n✓ Saved: {individual_path}")
                print(f"  Rows: {len(model_results)}")
            else:
                failed_models.append(model_name)
                print(f"\n✗ No results for {model_name}")

        except Exception as e:
            failed_models.append(model_name)
            print(f"\n✗ Error evaluating {model_name}: {e}")
            import traceback
            traceback.print_exc()

    # Combine and save all results
    if all_model_results:
        print("\n" + "="*80)
        print("SAVING COMBINED RESULTS")
        print("="*80)

        combined_df = pd.concat(all_model_results, ignore_index=True)

        # Save combined results
        combined_path = os.path.join(
            output_dir,
            f'all_models_cross_noise_{timestamp}.csv'
        )
        combined_df.to_csv(combined_path, index=False)
        print(f"\n✓ Combined results: {combined_path}")
        print(f"  Total rows: {len(combined_df)}")
        print(f"  Models: {combined_df['Model'].nunique()}")

        # Generate summary
        print("\n" + "="*80)
        print("SUMMARY BY MODEL AND EXPERIMENT")
        print("="*80)

        summary = combined_df.groupby(['Model', 'Experiment']).agg({
            'Accuracy': ['mean', 'std', 'count']
        }).round(2)

        print(summary.to_string())

        # Save summary
        summary_path = os.path.join(
            output_dir,
            f'summary_cross_noise_{timestamp}.csv'
        )
        summary.to_csv(summary_path)
        print(f"\n✓ Summary saved: {summary_path}")

        # Print pivot table
        print("\n" + "="*80)
        print("AVERAGE ACCURACY BY MODEL AND EXPERIMENT")
        print("="*80)

        pivot = combined_df.pivot_table(
            values='Accuracy',
            index='Model',
            columns='Experiment',
            aggfunc='mean'
        ).round(2)

        print(pivot.to_string())

        # Save pivot table
        pivot_path = os.path.join(
            output_dir,
            f'pivot_cross_noise_{timestamp}.csv'
        )
        pivot.to_csv(pivot_path)
        print(f"\n✓ Pivot table saved: {pivot_path}")

    # Final summary
    print("\n" + "="*80)
    print("EVALUATION COMPLETE!")
    print("="*80)
    print(f"Successful models: {len(successful_models)}/{len(models)}")
    for model in successful_models:
        print(f"  ✓ {model}")
    if failed_models:
        print(f"\nFailed models: {len(failed_models)}/{len(models)}")
        for model in failed_models:
            print(f"  ✗ {model}")
    print(f"\nResults saved to: {output_dir}")
    print("="*80)


# Fix the function name typo
def compare_cross_noise(model, source_df, target_df, distractors_df, main_col, versions, prefix=""):
    """Wrapper function with correct name."""
    return compare_languages_cross_noise(model, source_df, target_df, distractors_df, main_col, versions, prefix)


if __name__ == "__main__":
    main()

Using device: cuda

CLSD CROSS-NOISE EVALUATION
Timestamp: 20251102_153228
Models to evaluate: 4
Tasks per model: 2
Device: cuda

New Experiments:
  1. BL/SD→SnP (Both noisy) - BL query, SnP target+distractors
  2. BL/SD→BL/SD (SnP distractors) - BL query+target, SnP distractors
  3. SnP→BL/SD (Both noisy) - SnP query, BL target+distractors
  4. SnP→SnP (BL distractors) - SnP query+target, BL distractors


################################################################################
# MODEL 1/4: impresso-project/halloween_workshop_ocr_robust_with_lux_preview
################################################################################

MODEL: impresso-project/halloween_workshop_ocr_robust_with_lux_preview


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/283 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

✓ Loaded with trust_remote_code=True
✓ Moved to device: cuda

[Task 1/2] Cross-Noise BL/SD-SnP - CLSD-wmt19
--------------------------------------------------------------------------------
  Found levels: clean=clean, BL=bl-distorted, SnP=SaltnPepper

Cross-noise experiments: 4 configurations × 2 languages = 8 evaluations


✓ Completed 8 cross-noise evaluations

[Task 2/2] Cross-Noise BL/SD-SnP - CLSD-wmt21
--------------------------------------------------------------------------------
  Found levels: clean=clean, BL=bl-distorted, SnP=SaltnPepper

Cross-noise experiments: 4 configurations × 2 languages = 8 evaluations


✓ Completed 8 cross-noise evaluations

✓ Saved: /content/drive/MyDrive/OCR/results/Alibaba_NLP_gte_multilingual_base_cross_noise_20251102_153228.csv
  Rows: 16

SAVING COMBINED RESULTS

✓ Combined results: /content/drive/MyDrive/OCR/results/all_models_cross_noise_20251102_153228.csv
  Total rows: 64
  Models: 4

SUMMARY BY MODEL AND EXPERIMENT
                                                                                              Accuracy            
                                                                                                  mean   std count
Model                                                           Experiment                                        
Alibaba-NLP/gte-multilingual-base                               BL/SD→BL/SD (SnP distractors)    76.26  3.38     4
                                                                BL/SD→SnP (Both noisy)           79.08  0.83     4
                                                                SnP→BL/SD (Both

### X-STS

In [ ]:
import os
import pandas as pd
from sentence_transformers import SentenceTransformer, util
import torch
from tqdm import tqdm
from scipy.stats import spearmanr


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

def find_model_directories(prefix):
    # NEW: if it's a Hugging Face model id, pass it straight through
    if "/" in prefix:                         # e.g. "Alibaba-NLP/gte-multilingual-base"
        return [prefix]

    # model_base_dir = os.path.join(OCR_DIR, "trained_models")
    model_base_dir = os.path.join(os.getcwd(), "trained_models")

    if not os.path.exists(model_base_dir):
        print(f"[find_model_directories] Base dir does not exist: {model_base_dir}")
        return []
    return [
        os.path.join(model_base_dir, d)
        for d in os.listdir(model_base_dir)
        if d.startswith(prefix) and os.path.isdir(os.path.join(model_base_dir, d))
    ]

def evaluate_sts_task(model, sts_es_en, sts_ar_en, sts_tr_en, device, use_query_prefix=False):
    def add_prefix(texts, prefix):
        return [f"{prefix}{text}" for text in texts]

    def process_sts_data(sentences1, sentences2, scores, prefix=""):
        if prefix:
            sentences1 = add_prefix(sentences1, prefix)
            sentences2 = add_prefix(sentences2, prefix)
        embeddings1 = model.encode(sentences1, convert_to_tensor=True, device=device)
        embeddings2 = model.encode(sentences2, convert_to_tensor=True, device=device)
        cosine_scores = util.cos_sim(embeddings1, embeddings2).diag().cpu().numpy()
        return spearmanr(cosine_scores, scores).correlation * 100

    spearman_corr_es_en = process_sts_data(
        sts_es_en['es'].tolist(), sts_es_en['en'].tolist(), sts_es_en['similarity_score'].tolist(),
        "query: " if use_query_prefix else ""
    )
    spearman_corr_ar_en = process_sts_data(
        sts_ar_en['ar'].tolist(), sts_ar_en['en'].tolist(), sts_ar_en['similarity_score'].tolist(),
        "query: " if use_query_prefix else ""
    )
    spearman_corr_tr_en = process_sts_data(
        sts_tr_en['tr'].tolist(), sts_tr_en['en'].tolist(), sts_tr_en['similarity_score'].tolist(),
        "query: " if use_query_prefix else ""
    )

    avg_spearman_corr = (spearman_corr_es_en + spearman_corr_ar_en + spearman_corr_tr_en) / 3
    return spearman_corr_es_en, spearman_corr_ar_en, spearman_corr_tr_en, avg_spearman_corr

def process_and_save_results(tasks, sts_es_en, sts_ar_en, sts_tr_en):
    for task in tasks:
        results = []
        task_name = task['name']
        prefixes = task['prefixes']

        print("Processing Task Name:", task_name)
        save_name = task_name.replace(' ', '-') + "-sts-evaluations.csv"

        for model_prefix in tqdm(prefixes, desc=f"Processing Task {task_name}"):
            model_dirs = find_model_directories(model_prefix)
            if not model_dirs:
                print(f"No models found for prefix '{model_prefix}'")
                continue

            prefix_results = []
            for model_dir in model_dirs:
                try:
                    model = SentenceTransformer(model_dir, trust_remote_code=True)
                    model.to(device)
                except Exception as e:
                    print(f"Error loading model '{model_dir}': {e}")
                    continue

                spearman_es_en, spearman_ar_en, spearman_tr_en, avg_spearman = evaluate_sts_task(
                    model, sts_es_en, sts_ar_en, sts_tr_en, device, use_query_prefix=True
                )

                prefix_results.append({
                    'Model': model_dir,
                    'Task': task_name,
                    'Spearman ES-EN': spearman_es_en,
                    'Spearman AR-EN': spearman_ar_en,
                    'Spearman TR-EN': spearman_tr_en,
                    'Average Spearman': avg_spearman
                })

            prefix_results_df = pd.DataFrame(prefix_results)
            if not prefix_results_df.empty:
                avg_results = prefix_results_df.groupby(['Task'])[
                    ['Spearman ES-EN', 'Spearman AR-EN', 'Spearman TR-EN', 'Average Spearman']
                ].mean().reset_index()
                avg_results['Model Prefix'] = model_prefix
                results.extend(avg_results.to_dict('records'))

        results_df = pd.DataFrame(results)
        results_df.to_csv(save_name, index=False)
        print(f"Results for task '{task_name}' saved to {save_name}")

if __name__ == '__main__':
    # STS evaluation data (fixed paths)
    sts_es_en = pd.read_csv(f'{OCR_DIR}/inputs/sts17_es-en.csv')
    sts_ar_en = pd.read_csv(f'{OCR_DIR}/inputs/sts17_ar-en.csv')
    sts_tr_en = pd.read_csv(f'{OCR_DIR}/inputs/sts17_tr-en.csv')

    # Models to test: local families (prefixes) + your 3 HF models
    tasks = [
        {
            'name': 'All-Local-and-HF',
            'prefixes': [
                "impresso-project_histlux-gte-multilingual-base+TED_Impresso_SUMDOCS_SUMMARY-steps3000-bs8-seed100-20251022-110331",
                "Alibaba-NLP_gte-multilingual-base+TED_Impresso_SUMDOCS_SUMMARY-steps3000-bs8-seed100-20251022-112015"
                # ---- Local families (prefix basenames) ----
                # "Alibaba-NLP_gte-multilingual-base+TED",
                # "Alibaba-NLP_gte-multilingual-base+TED_Impresso",
                # "Alibaba-NLP_gte-multilingual-base+TED_Impresso_SUMDOCS",
                # "Alibaba-NLP_gte-multilingual-base+TED_Impresso_SUMDOCS_SUMMARY",

                # "impresso-project_histlux-gte-multilingual-base+TED",
                # "impresso-project_histlux-gte-multilingual-base+TED_Impresso",
                # "impresso-project_histlux-gte-multilingual-base+TED_Impresso_SUMDOCS",
                # "impresso-project_histlux-gte-multilingual-base+TED_Impresso_SUMDOCS_SUMMARY",

                # "trained_Alibaba-NLP+TED_Impresso+LUX",
                # "trained_Alibaba-NLP+TED_Impresso_SUMDOCS+LUX",
                # "trained_Alibaba-NLP+TED_Impresso_SUMDOCS_SUMMARY",
                # "trained_Alibaba-NLP-+TED+LUX",

                # # ---- Hugging Face models (exact IDs) ----
                # "Alibaba-NLP/gte-multilingual-base",
                # "impresso-project/histlux-gte-multilingual-base",
                # "impresso-project/ocr-quality-assessor-unigram-light",
            ]
        },
    ]

    process_and_save_results(tasks, sts_es_en, sts_ar_en, sts_tr_en)


Using device: cuda
Processing Task Name: All-Local-and-HF


Processing Task All-Local-and-HF:   0%|          | 0/2 [00:00<?, ?it/s]The module name impresso_hyphen_project_histlux_hyphen_gte_hyphen_multilingual_hyphen_base+TED_Impresso_SUMDOCS_SUMMARY_hyphen_steps3000_hyphen_bs8_hyphen_seed100_hyphen_20251022_hyphen_110331 (originally impresso-project_histlux-gte-multilingual-base+TED_Impresso_SUMDOCS_SUMMARY-steps3000-bs8-seed100-20251022-110331) is not a valid Python identifier. Please rename the original module to avoid import issues.
The module name impresso_hyphen_project_histlux_hyphen_gte_hyphen_multilingual_hyphen_base+TED_Impresso_SUMDOCS_SUMMARY_hyphen_steps3000_hyphen_bs8_hyphen_seed100_hyphen_20251022_hyphen_110331 (originally impresso-project_histlux-gte-multilingual-base+TED_Impresso_SUMDOCS_SUMMARY-steps3000-bs8-seed100-20251022-110331) is not a valid Python identifier. Please rename the original module to avoid import issues.
The module name impresso_hyphen_project_histlux_hyphen_gte_hyphen_multilingual_hyphen_base+TED_Impresso_S

Results for task 'All-Local-and-HF' saved to All-Local-and-HF-sts-evaluations.csv


### MLDR

In [ ]:
!pip install rank_bm25

In [ ]:
import os, math
import numpy as np
from datasets import load_dataset
from sentence_transformers import SentenceTransformer
from rank_bm25 import BM25Okapi

# Parameters
model_dir_prefixes = ["Alibaba-NLP_gte-multilingual-base",
                      "histlux-gte-multilingual-base"
                      ]
# model_dir_prefix = "Alibaba-NLP_gte-multilingual-base"
# model_dir_prefix = "histlux-gte-multilingual-base"

for model_dir_prefix in model_dir_prefixes:
  languages = ["en", "de", "fr","tu","ru","es","it"]
  # max_length = 512

  models_root = "./trained_models"
  model_dirs = [os.path.join(models_root, d) for d in os.listdir(models_root)
                if d.startswith(model_dir_prefix) and os.path.isdir(os.path.join(models_root, d))]

  header = "Model"
  for lang in languages:
      header += f", {lang}_NDCG@10, {lang}_MRR"
  out_lines = [header]

  # Helper: compute NDCG@10 for a list of ranks (1-indexed)
  def ndcg_at_10(ranks):
      # If rank > 10 (not retrieved in top10), contribution is 0
      # NDCG@10 for a single query with a single relevant item at 'rank':
      if ranks > 10:
          return 0.0
      # relevance = 1 for the one relevant document
      return 1.0 / math.log2(ranks + 1)

  for model_path in model_dirs + ["BM25_BASELINE"]:
      model_name = os.path.basename(model_path) if model_path != "BM25_BASELINE" else "BM25_baseline"
      print(f"Evaluating model: {model_name}")
      metrics = {lang: {"ndcg10": 0.0, "mrr": 0.0} for lang in languages}
      total_queries = {lang: 0 for lang in languages}

      # If model_path is a real model, load it; if it's BM25 baseline, we will handle separately
      model = None
      if model_path != "BM25_BASELINE":
          model = SentenceTransformer(model_path, device=('cuda' if torch.cuda.is_available() else 'cpu'),trust_remote_code=True)
          # model.max_seq_length = max_length  # set max length for encoding

      for lang in languages:
          # Load MLDR data for this language
          data = load_dataset("Shitao/MLDR", lang)
          # The dataset has 'train', 'dev', 'test' splits and also a 'corpus' subset:
          test_queries = data['test']       # list of {'query_id', 'query', 'positive_passages': [...], 'negative_passages': [...]}
          corpus = data[f"corpus-{lang}"]   # list of {'docid', 'text'} for all documents in the corpus

          # Prepare corpus documents and a lookup for docid -> index
          corpus_texts = [doc["text"] for doc in corpus]
          docid_to_index = {doc["docid"]: idx for idx, doc in enumerate(corpus)}

          # Encode corpus documents with embedding model or prepare BM25 index
          corpus_embeddings = None
          bm25 = None
          if model is not None:
              # Embed all documents in corpus (potentially large, so we do in batches)
              corpus_embeddings = model.encode(corpus_texts, batch_size=16, convert_to_numpy=True)
              # Normalize embeddings for cosine similarity
              corpus_embeddings = corpus_embeddings / np.linalg.norm(corpus_embeddings, axis=1, keepdims=True)
          else:
              # BM25 baseline: initialize inverted index
              tokenized_docs = [text.split() for text in corpus_texts]  # simple whitespace tokenization
              bm25 = BM25Okapi(tokenized_docs)

          # Evaluate each query
          ndcg_sum = 0.0
          mrr_sum = 0.0
          for query in test_queries:
              q_text = query["query"]
              # The positive passage list contains the relevant doc (one entry in test set)
              pos_docid = query["positive_passages"][0]["docid"]
              relevant_index = docid_to_index[pos_docid]

              # Get similarity scores for all docs
              if model is not None:
                  q_emb = model.encode(q_text, convert_to_numpy=True)
                  q_emb = q_emb / np.linalg.norm(q_emb)  # normalize
                  # Compute cosine similarities between query and all corpus embeddings
                  scores = np.dot(corpus_embeddings, q_emb)  # cosine similarity since both normalized
              else:
                  # BM25: get relevance scores for query
                  scores = bm25.get_scores(q_text.split())
              # Rank documents by score (higher is more similar/relevant)
              ranked_indices = np.argsort(-scores)  # indices sorted by descending score
              rank_of_relevant = int(np.where(ranked_indices == relevant_index)[0][0]) + 1  # 1-indexed rank

              # Compute metrics
              total_queries[lang] += 1
              # NDCG@10: only one relevant doc, calculate based on its rank
              ndcg_val = 0.0
              if rank_of_relevant <= 10:
                  ndcg_val = 1.0 / math.log2(rank_of_relevant + 1)
              ndcg_sum += ndcg_val
              # Reciprocal rank
              mrr_sum += 1.0 / rank_of_relevant

          # Average metrics for this language
          if total_queries[lang] > 0:
              metrics[lang]["ndcg10"] = ndcg_sum / total_queries[lang]
              metrics[lang]["mrr"] = mrr_sum / total_queries[lang]
          print(f"  [{lang}] NDCG@10 = {metrics[lang]['ndcg10']:.4f}, MRR = {metrics[lang]['mrr']:.4f}")

      # Append to results
      line = model_name
      for lang in languages:
          line += f", {metrics[lang]['ndcg10']:.4f}, {metrics[lang]['mrr']:.4f}"
      out_lines.append(line)

  # Save to CSV
  with open(f"{OCR_DIR}/results/mldr_results.csv", "w", encoding="utf-8") as fout:
      fout.write("\n".join(out_lines))

  print("MLDR evaluation completed. Results in mldr_results.csv.")


Evaluating model: Alibaba-NLP_gte-multilingual-base+TED+LUX+SUMALL+DEFR-taskbatches-DE-FR-TED-steps3000-bs8-seed777-20250925-125232


README.md: 0.00B [00:00, ?B/s]

MLDR.py: 0.00B [00:00, ?B/s]

RuntimeError: Dataset scripts are no longer supported, but found MLDR.py

### HISTU

In [ ]:
!pip install Levenshtein

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 159.9/159.9 kB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 61.0 MB/s eta 0:00:00


In [ ]:
import os
import pandas as pd
from sentence_transformers import SentenceTransformer
import torch
import numpy as np
import json
from tqdm import tqdm
from scipy.spatial.distance import cdist
from pathlib import Path

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# def find_model_directories(prefix):
#     model_base_dir = "./trained_models"
#     return [os.path.join(model_base_dir, d) for d in os.listdir(model_base_dir) if d.startswith(prefix)]


def find_model_directories(prefix: str, base_dir: str = "./trained_models"):

    if "/" in prefix:
        return [prefix]

    base = Path(base_dir)
    if not base.exists():
        print(f"[find_model_directories] Base dir does not exist: {base_dir}")
        return []

    dirs = []
    for p in base.iterdir():
        name = p.name
        if not name.startswith(prefix):
            continue
        if p.is_dir():
            has_st_files = any((p / f).exists() for f in ["config.json", "modules.json"])
            if has_st_files:
                dirs.append(str(p))
            else:
                print(f"[skip] {p} matches prefix but is missing config/modules.json")
        else:
            # Helpful debug if a zip is shadowing your prefix
            if name.endswith(".zip"):
                print(f"[skip zip] {p} (zip files are ignored)")

    dirs.sort()
    print(f"[find_model_directories] prefix='{prefix}' -> {dirs}")
    return dirs

def numpy_cosine_similarity(v1, v2):
    v1 = np.squeeze(v1)
    v2 = np.squeeze(v2)
    return np.dot(v1, v2) / (np.linalg.norm(v1) * np.linalg.norm(v2))

def evaluate_historical_bitext_mining_task(bitext_data, model, sentence_embeddings=None):
    if sentence_embeddings is None:
        unique_sentences = set(
            sentence for entry in bitext_data for sentence in [entry["source_sentence"]] + entry["candidates"]
        )
        sentence_embeddings = {sentence: model.encode(sentence) for sentence in unique_sentences}

    correct_count = 0

    for entry in bitext_data:
        source_embedding = sentence_embeddings[entry["source_sentence"]]
        parallel_embedding = sentence_embeddings[entry["candidates"][0]]
        parallel_similarity = numpy_cosine_similarity([source_embedding], [parallel_embedding])

        candidate_embeddings = [sentence_embeddings[candidate] for candidate in entry["candidates"][1:]]

        similarities = 1 - cdist(candidate_embeddings, source_embedding.reshape(1, -1), metric="cosine").flatten()

        if np.max(similarities) < parallel_similarity:
            correct_count += 1

    accuracy = round(correct_count / len(bitext_data) * 100, 2)
    return accuracy, sentence_embeddings

def load_dataset(file_path):
    """
    Loads a dataset (list of dictionaries) from a JSONL file.

    Args:
        file_path (str): Path to the JSONL file.

    Returns:
        list: List of dictionaries loaded from the file.
    """
    dataset = []
    with open(file_path, "r") as file:
        for line in file:
            dataset.append(json.loads(line))
    return dataset

def process_and_save_results(tasks, bitext_files):
    results = []

    for task in tasks:
        task_name = task['name']
        prefixes = task['prefixes']
        print("Processing Task Name: " + task_name)
        save_name = f"{OCR_DIR}/results/{task_name.replace(' ', '-')}-bitext-evaluations_x_mono.csv"

        for model_prefix in tqdm(prefixes, desc=f"Processing Task {task_name}"):
            model_dirs = find_model_directories(model_prefix)
            if not model_dirs:
                print(f"No models found for prefix '{model_prefix}'")
                continue

            prefix_results = []
            # save_name = f"{OCR_DIR}/results/{model_dirs}_{task_name.replace(' ', '-')}-bitext-evaluations_x_mono.csv"

            for model_dir in model_dirs:
                try:
                    model = SentenceTransformer(model_dir, trust_remote_code=True)
                    model.to(device)
                except ValueError as e:
                    print(f"Error loading model '{model_dir}': {e}")
                    continue

                # Evaluate the model on each bitext dataset
                accuracy_results = {}
                for bitext_name, bitext_data in bitext_files.items():
                    accuracy, _ = evaluate_historical_bitext_mining_task(
                        bitext_data, model
                    )
                    accuracy_results[f'{bitext_name} Accuracy'] = accuracy

                prefix_results.append({
                    'Model': model_dir,
                    'Task': task_name,
                    **accuracy_results
                })

            prefix_results_df = pd.DataFrame(prefix_results)
            if not prefix_results_df.empty:
                accuracy_columns = [col for col in prefix_results_df.columns if 'Accuracy' in col]
                avg_results = prefix_results_df.groupby(['Task'])[accuracy_columns].mean().reset_index()
                avg_results['Model Prefix'] = model_prefix
                results.extend(avg_results.to_dict('records'))

        # Save averaged results for this task
        results_df = pd.DataFrame(results)
        results_df.to_csv(save_name, index=False)
        print(f"Results for task '{task_name}' and model prefix saved to {save_name}")

if __name__ == '__main__':
    # Define tasks here
    tasks = [


    # ---------------- Hugging Face models ----------------

    # {
    #     'name': 'HF Alibaba GTE Base',
    #     'prefixes': ['Alibaba-NLP/gte-multilingual-base'],
    # },
    # {
    #     'name': 'HF Impresso histlux-gte',
    #     'prefixes': ['impresso-project/histlux-gte-multilingual-base'],
    # },
    # {
    #     'name': 'HF OCR Quality Assessor (unigram-light)',
    #     'prefixes': ['impresso-project/ocr-quality-assessor-unigram-light'],
    # },


    # ----------------- Alibaba family -----------------
    # {
    #     'name': 'Alibaba TED',
    #     'prefixes': ["Alibaba-NLP_gte-multilingual-base+TED"],
    # },
    # {
    #     'name': 'Alibaba TED_Impresso',
    #     'prefixes': ["Alibaba-NLP_gte-multilingual-base+TED_Impresso"],
    # },
    # {
    #     'name': 'Alibaba TED_Impresso_SUMDOCS',
    #     'prefixes': ["Alibaba-NLP_gte-multilingual-base+TED_Impresso_SUMDOCS"],
    # },
    # {
    #     'name': 'Alibaba TED_Impresso_SUMDOCS_SUMMARY',
    #     'prefixes': ["Alibaba-NLP_gte-multilingual-base+TED_Impresso_SUMDOCS_SUMMARY"],
    # },

    # ------------- impresso-project family -------------
    # {
    #     'name': 'Impresso TED',
    #     'prefixes': ["impresso-project_histlux-gte-multilingual-base+TED"],
    # },
    # {
    #     'name': 'Impresso TED_Impresso',
    #     'prefixes': ["impresso-project_histlux-gte-multilingual-base+TED_Impresso"],
    # },
    # {
    #     'name': 'Impresso TED_Impresso_SUMDOCS',
    #     'prefixes': ["impresso-project_histlux-gte-multilingual-base+TED_Impresso_SUMDOCS"],
    # },
    # {
    #     'name': 'Impresso TED_Impresso_SUMDOCS_SUMMARY',
    #     'prefixes': ["impresso-project_histlux-gte-multilingual-base+TED_Impresso_SUMDOCS_SUMMARY"],
    # },

    # ---------------- trained_* (+LUX) -----------------
    # {
    #     'name': 'trained Alibaba TED_Impresso +LUX',
    #     'prefixes': ["trained_Alibaba-NLP+TED_Impresso+LUX"],
    # },
    # {
    #     'name': 'trained Alibaba TED_Impresso_SUMDOCS +LUX',
    #     'prefixes': ["trained_Alibaba-NLP+TED_Impresso_SUMDOCS+LUX"],
    # },
    # {
    #     'name': 'trained Alibaba TED_Impresso_SUMDOCS_SUMMARY',
    #     'prefixes': ["trained_Alibaba-NLP+TED_Impresso_SUMDOCS_SUMMARY"],
    # },
    # {
    #     'name': 'trained Alibaba TED +LUX',
    #     'prefixes': ["trained_Alibaba-NLP-+TED+LUX"],
    # },
    {
        'name': 'trained Alibaba TED_Impresso_SUMDOCS_SUMMARY',
        'prefixes': ["Alibaba-NLP_gte-multilingual-base+TED_Impresso_SUMDOCS_SUMMARY"],
    },
    {
        'name': 'LUX_finetuned',
        'prefixes': ["impresso-project_histlux-gte-multilingual-base+TED_Impresso_SUMDOCS_SUMMARY"],
    },
]

    # Define bitext data files for different evaluations
    bitext_files = {
        'Hist BIM DE -> LB' : load_dataset(f'{OCR_DIR}/evaluation_sets/bitext_mining_task_de_to_lb.jsonl'),
        'Hist BIM LB -> DE' : load_dataset(f'{OCR_DIR}/evaluation_sets/bitext_mining_task_lb_to_de.jsonl'),
        'Hist BIM FR -> LB' : load_dataset(f'{OCR_DIR}/evaluation_sets/bitext_mining_task_fr_to_lb.jsonl'),
        'Hist BIM LB -> FR' : load_dataset(f'{OCR_DIR}/evaluation_sets/bitext_mining_task_lb_to_fr.jsonl'),
	      'Hist BIM EN -> LB' : load_dataset(f'{OCR_DIR}/evaluation_sets/bitext_mining_task_en_to_lb.jsonl'),
        'Hist BIM LB -> EN' : load_dataset(f'{OCR_DIR}/evaluation_sets/bitext_mining_task_lb_to_en.jsonl'),
    }

    # Run the evaluation process
    process_and_save_results(tasks, bitext_files)


Using device: cuda
Processing Task Name: trained Alibaba TED_Impresso_SUMDOCS_SUMMARY


Processing Task trained Alibaba TED_Impresso_SUMDOCS_SUMMARY:   0%|          | 0/1 [00:00<?, ?it/s]The module name Alibaba_hyphen_NLP_gte_hyphen_multilingual_hyphen_base+TED_Impresso_SUMDOCS_SUMMARY (originally Alibaba-NLP_gte-multilingual-base+TED_Impresso_SUMDOCS_SUMMARY) is not a valid Python identifier. Please rename the original module to avoid import issues.
The module name Alibaba_hyphen_NLP_gte_hyphen_multilingual_hyphen_base+TED_Impresso_SUMDOCS_SUMMARY (originally Alibaba-NLP_gte-multilingual-base+TED_Impresso_SUMDOCS_SUMMARY) is not a valid Python identifier. Please rename the original module to avoid import issues.
The module name Alibaba_hyphen_NLP_gte_hyphen_multilingual_hyphen_base+TED_Impresso_SUMDOCS_SUMMARY (originally Alibaba-NLP_gte-multilingual-base+TED_Impresso_SUMDOCS_SUMMARY) is not a valid Python identifier. Please rename the original module to avoid import issues.
The module name Alibaba_hyphen_NLP_gte_hyphen_multilingual_hyphen_base+TED_Impresso_SUMDOCS_SUMMA

[find_model_directories] prefix='Alibaba-NLP_gte-multilingual-base+TED_Impresso_SUMDOCS_SUMMARY' -> ['trained_models/Alibaba-NLP_gte-multilingual-base+TED_Impresso_SUMDOCS_SUMMARY']


Processing Task trained Alibaba TED_Impresso_SUMDOCS_SUMMARY: 100%|██████████| 1/1 [06:27<00:00, 387.45s/it]


Results for task 'trained Alibaba TED_Impresso_SUMDOCS_SUMMARY' and model prefix saved to /content/drive/MyDrive/OCR//results/trained-Alibaba-TED_Impresso_SUMDOCS_SUMMARY-bitext-evaluations_x_mono.csv
Processing Task Name: LUX_finetuned


Processing Task LUX_finetuned:   0%|          | 0/1 [00:00<?, ?it/s]The module name impresso_hyphen_project_histlux_hyphen_gte_hyphen_multilingual_hyphen_base+TED_Impresso_SUMDOCS_SUMMARY (originally impresso-project_histlux-gte-multilingual-base+TED_Impresso_SUMDOCS_SUMMARY) is not a valid Python identifier. Please rename the original module to avoid import issues.
The module name impresso_hyphen_project_histlux_hyphen_gte_hyphen_multilingual_hyphen_base+TED_Impresso_SUMDOCS_SUMMARY (originally impresso-project_histlux-gte-multilingual-base+TED_Impresso_SUMDOCS_SUMMARY) is not a valid Python identifier. Please rename the original module to avoid import issues.
The module name impresso_hyphen_project_histlux_hyphen_gte_hyphen_multilingual_hyphen_base+TED_Impresso_SUMDOCS_SUMMARY (originally impresso-project_histlux-gte-multilingual-base+TED_Impresso_SUMDOCS_SUMMARY) is not a valid Python identifier. Please rename the original module to avoid import issues.
The module name impresso_hyph

[find_model_directories] prefix='impresso-project_histlux-gte-multilingual-base+TED_Impresso_SUMDOCS_SUMMARY' -> ['trained_models/impresso-project_histlux-gte-multilingual-base+TED_Impresso_SUMDOCS_SUMMARY']


Processing Task LUX_finetuned: 100%|██████████| 1/1 [06:17<00:00, 377.47s/it]

Results for task 'LUX_finetuned' and model prefix saved to /content/drive/MyDrive/OCR//results/LUX_finetuned-bitext-evaluations_x_mono.csv


## PARALUX

In [ ]:
from datasets import load_dataset
from sklearn.metrics.pairwise import cosine_similarity
import torch
import numpy as np


In [ ]:
# Load the ParaLux test dataset
paralux_test = load_dataset("fredxlpy/ParaLux", split="test")

# Now you can work with the paralux_test dataset
paralux_data = {}
paralux_data['anchor'] = paralux_test['anchor']
paralux_data['positive'] = paralux_test['paraphrase']
paralux_data['negative'] = paralux_test['not_paraphrase']

README.md: 0.00B [00:00, ?B/s]

benchmark.json:   0%|          | 0.00/86.4k [00:00<?, ?B/s]

Generating test split:   0%|          | 0/312 [00:00<?, ? examples/s]

In [ ]:
# EVAL_ROOT = os.path.join(OCR_DIR,"trained_models")
EVAL_ROOT = os.path.join(os.getcwd(), "trained_models")

In [ ]:
def evaluate_paralux_performance(anchor, positive, negative, model):
    """
    Measures the frequency with which the model's embeddings yield a higher cosine similarity
    between anchor and positive than between anchor and negative examples.

    Parameters:
        anchor (list): List of anchor text strings.
        positive (list): List of positive/paraphrase text strings.
        negative (list): List of negative/not paraphrase text strings.
        model: A model with an `encode` method to generate text embeddings.

    Returns:
        float: The frequency of anchor-positive similarity being higher than anchor-negative similarity.
    """
    # Encode the texts
    anchor_embeddings = model.encode(anchor)
    positive_embeddings = model.encode(positive)
    negative_embeddings = model.encode(negative)

    # Compute cosine similarities
    anchor_positive_sim = cosine_similarity(anchor_embeddings, positive_embeddings).diagonal()
    anchor_negative_sim = cosine_similarity(anchor_embeddings, negative_embeddings).diagonal()

    # Calculate the percentage of samples where positive similarity > negative similarity
    higher_similarity_percentage = round(np.mean(anchor_positive_sim > anchor_negative_sim) * 100, 2)

    return higher_similarity_percentage

In [ ]:
def run_lux_embeddings_evaluations(
    model,
    model_name,
    similarity_data,
    run_paralux_clf=True,
):
    """
    Combines template-based model evaluation and similarity frequency measurement.

    Parameters:
        model: A model object with an `.encode()` method for generating embeddings.
        model_name (str): Name of the model for display in the results.
        dataset: The dataset containing examples with 'text' and 'category' fields (required for evaluation).
        class_to_templates: A dictionary mapping categories to their list of templates (required for evaluation).
        similarity_data (dict): A dictionary containing 'anchor', 'positive', and 'negative' lists for similarity check.
        run_evaluation (bool): Whether to run the template-based evaluation.
        run_similarity_check (bool): Whether to run the similarity frequency check.

    Returns:
        dict: A dictionary containing results for evaluation and similarity checks.
    """
    results = {}

    if run_paralux_clf:
        if not (similarity_data and
                "anchor" in similarity_data and
                "positive" in similarity_data and
                "negative" in similarity_data):
            raise ValueError("A dictionary containing 'anchor', 'positive', and 'negative' lists must be provided for similarity check.")

        paralux_performance = evaluate_paralux_performance(
            similarity_data["anchor"],
            similarity_data["positive"],
            similarity_data["negative"],
            model
        )
        results["PARALux Accuracy"] = paralux_performance

    return results

In [ ]:
# results_detailed = run_lux_embeddings_evaluations(model, model_name, sib_test_dataset, class_to_templates, similarity_data=paralux_data)


In [ ]:
import os, re, gc, shutil, traceback, difflib
from datetime import datetime
import pandas as pd
import torch
from sentence_transformers import SentenceTransformer

_ts_re = re.compile(r'(\d{8}-\d{6})(?!.*\d)')

def _boundary_prefix_match(name: str, prefix: str) -> bool:
    """
    True if `name` == prefix OR `name` starts with `prefix` followed by a dash.
    This prevents '...+TED' from matching '...+TED_Impresso'.
    """
    if not name.startswith(prefix):
        return False
    if len(name) == len(prefix):
        return True
    return name[len(prefix)] == '-'  # only accept "-steps..."


def _extract_ts(name: str):
    m = _ts_re.search(name)
    if not m: return None
    try: return datetime.strptime(m.group(1), "%Y%m%d-%H%M%S")
    except: return None

def _norm(s: str) -> str:
    return re.sub(r'[^a-z0-9]+', '', s.lower())

def _ensure_unzipped(zip_path: str, unzip_root: str) -> str:
    """Unpack a zip if needed and return the model dir containing modules.json."""
    base = os.path.splitext(os.path.basename(zip_path))[0]
    target_root = os.path.join(unzip_root, base)
    os.makedirs(target_root, exist_ok=True)
    # Only unpack if empty
    if not os.listdir(target_root):
        shutil.unpack_archive(zip_path, target_root)
    # If the archive created a subfolder, find the one with modules.json
    if os.path.exists(os.path.join(target_root, "modules.json")):
        return target_root
    for entry in os.listdir(target_root):
        p = os.path.join(target_root, entry)
        if os.path.isdir(p) and os.path.exists(os.path.join(p, "modules.json")):
            return p
    # Fallback: return root (SentenceTransformer will error if not a model folder)
    return target_root

def collect_model_targets(prefixes,
                          root,
                          include_hf_ids=True,
                          latest_per_prefix=True,
                          strict_prefix=True,      # <-- default to True now
                          include_zips=False,      # <-- all models already unzipped
                          debug=True,
                          suggest_topk=5):
    if not os.path.isdir(root):
        print(f"[warn] root does not exist or is not a dir: {root}")
        return []

    if debug:
        print("scanning root:", root)

    # gather only real dirs (ignore macOS ghost dirs)
    all_dirs = []
    for dirpath, dirnames, _ in os.walk(root):
        for d in dirnames:
            if d.startswith("._"):         # ignore macOS ghost entries
                continue
            all_dirs.append((d, os.path.join(dirpath, d)))

    matches = {}  # prefix -> list[(disp, path, ts, mtime, kind)]
    for p in prefixes:
        hits = []
        for d, full in all_dirs:
            ok = (_boundary_prefix_match(d, p) if strict_prefix
                  else (_norm(p) in _norm(d) or _norm(d).startswith(_norm(p))))
            if ok:
                ts = _extract_ts(d)
                try:
                    mtime = os.path.getmtime(full)
                except Exception:
                    mtime = 0
                hits.append((d, full, ts, mtime, "dir"))

        if hits:
            matches[p] = hits

    targets = []
    for p, items in matches.items():
        items_sorted = sorted(items, key=lambda x: (x[2] is not None, x[2], x[3]), reverse=True)
        if latest_per_prefix:
            disp, path, _, _, kind = items_sorted[0]
            targets.append((disp, path, kind))
        else:
            for disp, path, _, _, kind in items_sorted:
                targets.append((disp, path, kind))

    if include_hf_ids:
        for p in prefixes:
            if "/" in p:
                targets.append((p, p, "hf"))

    # de-dup keep order
    seen, final = set(), []
    for disp, path, kind in targets:
        key = (disp, path)
        if key not in seen:
            seen.add(key)
            final.append((disp, path, kind))

    if debug:
        print("\n[debug] matched targets:")
        for disp, path, kind in final:
            print(f" • {disp} ({kind}) -> {path}")

    return final


def evaluate_all_models(prefixes,
                        similarity_data,
                        root,
                        include_hf_ids=True,
                        latest_per_prefix=True,
                        strict_prefix=False,
                        include_zips=True,
                        unzip_root=None,
                        device=None,
                        save_csv_path=None):
    device = device or ("cuda" if torch.cuda.is_available() else "cpu")
    unzip_root = unzip_root or os.path.join(root, "_unzipped")
    os.makedirs(unzip_root, exist_ok=True)

    rows = []
    targets = collect_model_targets(prefixes,
                                    root=root,
                                    include_hf_ids=include_hf_ids,
                                    latest_per_prefix=latest_per_prefix,
                                    strict_prefix=strict_prefix,
                                    include_zips=include_zips,
                                    debug=True)

    if not targets:
        print("[warn] No model targets found.")
        return pd.DataFrame()

    print(f"\n[info] Will evaluate {len(targets)} models")
    for i, (disp, path_or_id, kind) in enumerate(targets, 1):
        print(f"\n[{i}/{len(targets)}] Loading: {path_or_id}")
        try:
            load_path = path_or_id
            if kind == "zip":
                load_path = _ensure_unzipped(path_or_id, unzip_root)

            model = SentenceTransformer(load_path, trust_remote_code=True).to(device)
            model.eval()
            with torch.no_grad():
                results = run_lux_embeddings_evaluations(
                    model=model,
                    model_name=disp,
                    similarity_data=similarity_data,
                    run_paralux_clf=True,
                )

            row = {
                "model_name": disp,
                "model_path_or_id": path_or_id,
                "load_kind": kind
            }
            row.update(results)
            rows.append(row)

        except Exception as e:
            print(f"[ERROR] {disp}: {e}")
            traceback.print_exc()
        finally:
            try: del model
            except: pass
            gc.collect()
            if torch.cuda.is_available():
                torch.cuda.empty_cache()

    df = pd.DataFrame(rows)
    metric_cols = [c for c in df.columns if c not in ("model_name", "model_path_or_id", "load_kind")]
    if metric_cols:
        df = df[["model_name", "model_path_or_id", "load_kind"] + metric_cols]
        if "PARALux Accuracy" in df.columns:
            df = df.sort_values("PARALux Accuracy", ascending=False)

    print("\n=== Evaluation Summary ===")
    with pd.option_context("display.max_columns", None, "display.width", 160):
        print(df.to_string(index=False))

    if save_csv_path:
        os.makedirs(os.path.dirname(save_csv_path) or ".", exist_ok=True)
        df.to_csv(save_csv_path, index=False)
        print(f"[saved] -> {save_csv_path}")

    return df


In [ ]:
ALL_PREFIXES = [
    # Alibaba family (exact)
    # "Alibaba-NLP_gte-multilingual-base+TED",
    # "Alibaba-NLP_gte-multilingual-base+TED_Impresso",
    # "Alibaba-NLP_gte-multilingual-base+TED_Impresso_SUMDOCS",
    # "Alibaba-NLP_gte-multilingual-base+TED_Impresso_SUMDOCS_SUMMARY",

    # impresso-project family (exact)
    # "impresso-project_histlux-gte-multilingual-base+TED",
    # "impresso-project_histlux-gte-multilingual-base+TED_Impresso",
    # "impresso-project_histlux-gte-multilingual-base+TED_Impresso_SUMDOCS",
    # "impresso-project_histlux-gte-multilingual-base+TED_Impresso_SUMDOCS_SUMMARY",

    # trained_* (no steps; keep full names)
    # "trained_Alibaba-NLP+TED_Impresso+LUX",
    # "trained_Alibaba-NLP+TED_Impresso_SUMDOCS+LUX",
    # "trained_Alibaba-NLP+TED_Impresso_SUMDOCS_SUMMARY",
    # "trained_Alibaba-NLP-+TED+LUX",  # odd, but keep as-is

    # ---------- Hugging Face model IDs ----------
    # "gte-multilingual-base",
    # "histlux-gte-multilingual-base",
    "Alibaba-NLP_gte-multilingual-base+TED_Impresso_SUMDOCS_SUMMARY-steps3000-bs8-seed100-20251022-112015",
    "impresso-project_histlux-gte-multilingual-base+TED_Impresso_SUMDOCS_SUMMARY-steps3000-bs8-seed100-20251022-110331"
]


In [ ]:
df_results = evaluate_all_models(
    prefixes=ALL_PREFIXES,
    similarity_data=paralux_data,
    root=EVAL_ROOT,
    include_hf_ids=True,     # keep if you also want HF IDs (those with '/')
    latest_per_prefix=True,  # one run per intended prefix
    strict_prefix=True,      # crucial to avoid prefix bleed
    include_zips=False,      # you said everything's unzipped
    save_csv_path=f"{OCR_DIR}/results/paralux_eval.csv",
)


The module name Alibaba_hyphen_NLP_gte_hyphen_multilingual_hyphen_base+TED_Impresso_SUMDOCS_SUMMARY_hyphen_steps3000_hyphen_bs8_hyphen_seed100_hyphen_20251022_hyphen_112015 (originally Alibaba-NLP_gte-multilingual-base+TED_Impresso_SUMDOCS_SUMMARY-steps3000-bs8-seed100-20251022-112015) is not a valid Python identifier. Please rename the original module to avoid import issues.
The module name Alibaba_hyphen_NLP_gte_hyphen_multilingual_hyphen_base+TED_Impresso_SUMDOCS_SUMMARY_hyphen_steps3000_hyphen_bs8_hyphen_seed100_hyphen_20251022_hyphen_112015 (originally Alibaba-NLP_gte-multilingual-base+TED_Impresso_SUMDOCS_SUMMARY-steps3000-bs8-seed100-20251022-112015) is not a valid Python identifier. Please rename the original module to avoid import issues.
The module name Alibaba_hyphen_NLP_gte_hyphen_multilingual_hyphen_base+TED_Impresso_SUMDOCS_SUMMARY_hyphen_steps3000_hyphen_bs8_hyphen_seed100_hyphen_20251022_hyphen_112015 (originally Alibaba-NLP_gte-multilingual-base+TED_Impresso_SUMDOCS_SU

scanning root: /content/trained_models

[debug] matched targets:
 • Alibaba-NLP_gte-multilingual-base+TED_Impresso_SUMDOCS_SUMMARY-steps3000-bs8-seed100-20251022-112015 (dir) -> /content/trained_models/Alibaba-NLP_gte-multilingual-base+TED_Impresso_SUMDOCS_SUMMARY-steps3000-bs8-seed100-20251022-112015
 • impresso-project_histlux-gte-multilingual-base+TED_Impresso_SUMDOCS_SUMMARY-steps3000-bs8-seed100-20251022-110331 (dir) -> /content/trained_models/impresso-project_histlux-gte-multilingual-base+TED_Impresso_SUMDOCS_SUMMARY-steps3000-bs8-seed100-20251022-110331

[info] Will evaluate 2 models

[1/2] Loading: /content/trained_models/Alibaba-NLP_gte-multilingual-base+TED_Impresso_SUMDOCS_SUMMARY-steps3000-bs8-seed100-20251022-112015


The module name impresso_hyphen_project_histlux_hyphen_gte_hyphen_multilingual_hyphen_base+TED_Impresso_SUMDOCS_SUMMARY_hyphen_steps3000_hyphen_bs8_hyphen_seed100_hyphen_20251022_hyphen_110331 (originally impresso-project_histlux-gte-multilingual-base+TED_Impresso_SUMDOCS_SUMMARY-steps3000-bs8-seed100-20251022-110331) is not a valid Python identifier. Please rename the original module to avoid import issues.
The module name impresso_hyphen_project_histlux_hyphen_gte_hyphen_multilingual_hyphen_base+TED_Impresso_SUMDOCS_SUMMARY_hyphen_steps3000_hyphen_bs8_hyphen_seed100_hyphen_20251022_hyphen_110331 (originally impresso-project_histlux-gte-multilingual-base+TED_Impresso_SUMDOCS_SUMMARY-steps3000-bs8-seed100-20251022-110331) is not a valid Python identifier. Please rename the original module to avoid import issues.
The module name impresso_hyphen_project_histlux_hyphen_gte_hyphen_multilingual_hyphen_base+TED_Impresso_SUMDOCS_SUMMARY_hyphen_steps3000_hyphen_bs8_hyphen_seed100_hyphen_202510


[2/2] Loading: /content/trained_models/impresso-project_histlux-gte-multilingual-base+TED_Impresso_SUMDOCS_SUMMARY-steps3000-bs8-seed100-20251022-110331

=== Evaluation Summary ===
                                                                                                       model_name                                                                                                                          model_path_or_id load_kind  PARALux Accuracy
impresso-project_histlux-gte-multilingual-base+TED_Impresso_SUMDOCS_SUMMARY-steps3000-bs8-seed100-20251022-110331 /content/trained_models/impresso-project_histlux-gte-multilingual-base+TED_Impresso_SUMDOCS_SUMMARY-steps3000-bs8-seed100-20251022-110331       dir              62.5
             Alibaba-NLP_gte-multilingual-base+TED_Impresso_SUMDOCS_SUMMARY-steps3000-bs8-seed100-20251022-112015              /content/trained_models/Alibaba-NLP_gte-multilingual-base+TED_Impresso_SUMDOCS_SUMMARY-steps3000-bs8-seed100-20251022-112015       

### OCR

In [ ]:
!pip install rank_bm25

In [ ]:
# eval_historic_ocr_xl_plus.py
import os, json, random, math
from itertools import islice
from typing import List, Dict, Optional, Tuple

import torch
import numpy as np
import pandas as pd
from tqdm import tqdm
from datasets import load_dataset
from sentence_transformers import SentenceTransformer, util
from rank_bm25 import BM25Okapi  # BM25 baseline

# ----------------------------
# Device / seeds
# ----------------------------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)
RNG_SEED = 42
random.seed(RNG_SEED); np.random.seed(RNG_SEED); torch.manual_seed(RNG_SEED)

# ----------------------------
# Eval config
# ----------------------------
EVAL_N = 1500                 # cap per language / dataset
MAX_CHUNK_LEN = 256           # clamped to model.max_seq_length
CHUNK_STRIDE = 128
BATCH_ENCODE = 16
TOPK = [1, 5, 10]
MIN_DOC_CHARS = 400
TITLE_MIN_CHARS = 10
OCR_BINS = 4                  # number of quantile bins for Europeana mean_ocr
MIN_BIN_SIZE = 150            # skip bins with fewer examples than this

# ----------------------------
# Helpers
# ----------------------------
def find_model_directories(prefix: str) -> List[str]:
    base = os.path.join(os.getcwd(), "trained_models")
    if not os.path.exists(base): return []
    return [os.path.join(base, d) for d in os.listdir(base)
            if os.path.isdir(os.path.join(base, d)) and d.startswith(prefix)]

def tag(text: str, model_name: str, kind: str) -> str:
    s = "" if text is None else str(text)
    low = model_name.lower()
    if "multilingual-e5" in low or "/e5" in low:
        return ("query: " if kind == "query" else "passage: ") + s
    return s

def _ensure_title(text: str, title: Optional[str]) -> str:
    if title and len(title.strip()) >= TITLE_MIN_CHARS:
        return title.strip()
    toks = (text or "").split()
    return " ".join(toks[:12]) if toks else "untitled"

# ----------------------------
# Chunking (no hard truncation)
# ----------------------------
def _word_chunks(words: List[str], chunk_len: int, stride: int) -> List[str]:
    out, i = [], 0
    while i < len(words):
        out.append(" ".join(words[i:i+chunk_len]))
        if i + chunk_len >= len(words): break
        i += max(1, stride)
    return out

def chunk_text_for_model(model: SentenceTransformer, text: str,
                         desired_len: int = MAX_CHUNK_LEN,
                         stride: int = CHUNK_STRIDE) -> List[str]:
    max_len = getattr(model, "max_seq_length", desired_len)
    chunk_len = min(desired_len, max_len - 2) if max_len and max_len > 2 else desired_len
    words = (text or "").split()
    if not words: return []
    return _word_chunks(words, chunk_len=chunk_len, stride=stride)

def encode_chunks_aggregate(model: SentenceTransformer, docs: List[str]) -> torch.Tensor:
    pooled = []
    for doc in tqdm(docs, desc="Encode docs (sliding windows)"):
        chunks = chunk_text_for_model(model, doc)
        if not chunks: chunks = [doc]
        embs = model.encode(chunks, batch_size=BATCH_ENCODE, convert_to_tensor=True,
                            device=device, show_progress_bar=False)
        pooled.append(embs.mean(dim=0))
    return torch.stack(pooled, dim=0) if pooled else torch.empty(0)

def encode_queries(model: SentenceTransformer, queries: List[str], model_name: str) -> torch.Tensor:
    q = [tag(q, model_name, "query") for q in queries]
    return model.encode(q, batch_size=BATCH_ENCODE, convert_to_tensor=True,
                        device=device, show_progress_bar=False)

# ----------------------------
# Dataset-specific loaders (schema-tailored)
# ----------------------------
def load_europeana_lang(lang_config: str, limit: int) -> List[Dict]:
    """
    Returns list of dicts: {id, lang, title, text, date, mean_ocr}
    """
    ds = load_dataset("biglam/europeana_newspapers", lang_config, split="train", streaming=True)
    rows, out = 0, []
    for ex in ds:
        text = ex.get("text") or ""
        if len(text) < MIN_DOC_CHARS: continue
        title = _ensure_title(text, ex.get("title"))
        mean_ocr = ex.get("mean_ocr")
        try:
            mean_ocr = float(mean_ocr) if mean_ocr is not None else None
        except Exception:
            mean_ocr = None
        out.append({
            "id": ex.get("id") or f"{lang_config}-{rows}",
            "lang": lang_config,
            "title": title,
            "text": text,
            "date": ex.get("date"),
            "mean_ocr": mean_ocr
        })
        rows += 1
        if rows >= limit: break
    return out

def load_americanstories(limit: int) -> List[Dict]:
    ds = load_dataset("dell-research-harvard/AmericanStories", "subset_years", split="train", streaming=True)
    rows, out = 0, []
    for ex in ds:
        text = ex.get("article") or ""
        if len(text) < MIN_DOC_CHARS: continue
        title = _ensure_title(text, ex.get("headline"))
        out.append({
            "id": ex.get("article_id") or f"as-{rows}",
            "lang": "en",
            "title": title,
            "text": text,
            "date": ex.get("date")
        })
        rows += 1
        if rows >= limit: break
    return out

def load_kubhist2(limit_docs: int, sentences_per_doc: int = 300) -> List[Dict]:
    ds = load_dataset("iguanodon-ai/kubhist2", split="train", streaming=True)
    out, cur_sent, cur_buf, doc_id = [], 0, [], 0
    for ex in ds:
        t = ex.get("text") or ""
        if not t: continue
        cur_buf.append(t.strip())
        cur_sent += 1
        if cur_sent >= sentences_per_doc:
            text = " ".join(cur_buf)
            if len(text) >= MIN_DOC_CHARS:
                out.append({
                    "id": f"kub-{doc_id}",
                    "lang": "sv",
                    "title": _ensure_title(text, None),
                    "text": text
                })
                doc_id += 1
                if doc_id >= limit_docs: break
            cur_buf, cur_sent = [], 0
    return out

# ----------------------------
# Pools & metrics
# ----------------------------
def build_qd_pool(examples: List[Dict]) -> Tuple[List[str], List[str], np.ndarray]:
    queries = [e["title"] for e in examples]
    docs = [e["text"] for e in examples]
    gold = np.arange(len(examples))
    return queries, docs, gold

def recall_at_k(scores: torch.Tensor, gold: np.ndarray, ks=TOPK) -> Dict[str, float]:
    N, M = scores.shape
    topk = torch.topk(scores, k=max(ks), dim=1).indices.cpu().numpy()
    out = {}
    for k in ks:
        hits = sum(gold[i] in topk[i, :k] for i in range(N))
        out[f"R@{k}"] = 100.0 * hits / N if N else 0.0
    out["P@1"] = out["R@1"]
    return out

# ----------------------------
# BM25 baseline (q→d)
# ----------------------------
def _tok(s: str) -> List[str]:
    return [w for w in (s or "").lower().split() if w]

def bm25_eval_qd(queries: List[str], docs: List[str], gold: np.ndarray) -> Dict[str, float]:
    tokenized_docs = [_tok(d) for d in docs]
    bm25 = BM25Okapi(tokenized_docs)
    topk = max(TOPK)
    hits_at_k = {k: 0 for k in TOPK}
    for i, q in enumerate(queries):
        scores = bm25.get_scores(_tok(q))
        # argsort descending
        idx = np.argsort(-np.array(scores))
        for k in TOPK:
            if gold[i] in idx[:k]:
                hits_at_k[k] += 1
    out = {f"R@{k}": 100.0 * hits_at_k[k] / len(queries) for k in TOPK} if queries else {f"R@{k}": 0.0 for k in TOPK}
    out["P@1"] = out["R@1"]
    return out

# ----------------------------
# Embedder evals
# ----------------------------
def eval_qd_embedder(model: SentenceTransformer, model_name: str, examples: List[Dict]) -> Dict[str, float]:
    q, d, gold = build_qd_pool(examples)
    qe = encode_queries(model, q, model_name)
    de = encode_chunks_aggregate(model, d)
    scores = util.cos_sim(qe, de)
    return recall_at_k(scores, gold)

def eval_dd_embedder(model: SentenceTransformer, examples: List[Dict]) -> Dict[str, float]:
    # split each doc in two halves; A retrieves B
    A, B = [], []
    for e in examples:
        words = e["text"].split()
        if len(words) < 200: continue
        mid = len(words)//2
        A.append(" ".join(words[:mid]))
        B.append(" ".join(words[mid:]))
    gold = np.arange(len(A))
    if not A:
        return {"P@1": 0.0, "R@5": 0.0, "R@10": 0.0}
    Ae = encode_chunks_aggregate(model, A)
    Be = encode_chunks_aggregate(model, B)
    scores = util.cos_sim(Ae, Be)
    return recall_at_k(scores, gold)

# ----------------------------
# Europeana cross-lingual (mixed pool)
# ----------------------------
def eval_europeana_xl_mixed(model: SentenceTransformer, model_name: str,
                            fr: List[Dict], de: List[Dict]) -> Dict[str, float]:
    fr_q = [e["title"] for e in fr]
    frd = [e["text"] for e in fr]
    ded = [e["text"] for e in de]
    pool = frd + ded
    gold_fr = np.arange(len(fr))  # FR gold at same index in pool
    qe_fr = encode_queries(model, fr_q, model_name)
    de_pool = encode_chunks_aggregate(model, pool)
    scores_fr = util.cos_sim(qe_fr, de_pool)
    fr_metrics = {f"FR→(FR+DE) {k}": v for k, v in recall_at_k(scores_fr, gold_fr).items()}

    de_q = [e["title"] for e in de]
    gold_de = np.arange(len(de)) + len(frd)  # DE docs offset
    qe_de = encode_queries(model, de_q, model_name)
    scores_de = util.cos_sim(qe_de, de_pool)
    de_metrics = {f"DE→(FR+DE) {k}": v for k, v in recall_at_k(scores_de, gold_de).items()}

    return {**fr_metrics, **de_metrics, "PoolSize": len(pool)}

# ----------------------------
# OCR-confidence stratification
# ----------------------------
def ocr_quantile_bins(examples: List[Dict], q_bins: int = OCR_BINS) -> List[Tuple[float, float]]:
  # mean_ocr per doc quality proxy
  #
    vals = [e["mean_ocr"] for e in examples if isinstance(e.get("mean_ocr"), (int, float))]
    qs = np.quantile(vals, np.linspace(0, 1, q_bins+1))
    bins = []
    for i in range(q_bins):
        lo, hi = float(qs[i]), float(qs[i+1])
        # left-closed, right-open for all but last; last is closed
        bins.append((lo, hi, i == q_bins-1))
    return bins

def filter_by_ocr_bin(examples: List[Dict], lo: float, hi: float) -> List[Dict]:
    out = []
    for e in examples:
        v = e.get("mean_ocr")
        if isinstance(v, (int, float)) and lo <= v <= hi:
            out.append(e)
    return out

# ----------------------------
# Orchestrator
# ----------------------------
def run_all(prefixes: List[str]):
    os.makedirs("./results", exist_ok=True)

    # Preload datasets
    eu_de = load_europeana_lang("de", EVAL_N)
    eu_fr = load_europeana_lang("fr", EVAL_N)
    eu_mix = (eu_de[:EVAL_N//2] + eu_fr[:EVAL_N//2])
    amer  = load_americanstories(EVAL_N)
    kub   = load_kubhist2(EVAL_N)

    # Precompute OCR bins for Europeana
    #
    de_bins = ocr_quantile_bins(eu_de, OCR_BINS)
    fr_bins = ocr_quantile_bins(eu_fr, OCR_BINS)

    all_rows = []
    for pref in prefixes:
        mdirs = find_model_directories(pref)
        if not mdirs:
            print(f"No models found for prefix '{pref}'");
            continue

        for mdir in mdirs:
            try:
                model = SentenceTransformer(mdir, trust_remote_code=True).to(device)
                model_name = os.path.basename(mdir)
            except Exception as e:
                print("Load failed:", mdir, e);
                continue

            # ---------------- Europeana (mono, full) ----------------
            def eval_qd_pair(examples, dataset_tag):
                q, d, gold = build_qd_pool(examples)
                # Embedder
                qd_emb = eval_qd_embedder(model, model_name, examples)
                # BM25
                qd_bm  = bm25_eval_qd(q, d, gold)
                # Δ
                delta_p1 = qd_emb["P@1"] - qd_bm["P@1"]
                return {
                    f"{dataset_tag} P@1(qd)": qd_emb["P@1"], f"{dataset_tag} R@5(qd)": qd_emb["R@5"], f"{dataset_tag} R@10(qd)": qd_emb["R@10"],
                    f"{dataset_tag} P@1(BM25)": qd_bm["P@1"], f"{dataset_tag} R@5(BM25)": qd_bm["R@5"], f"{dataset_tag} R@10(BM25)": qd_bm["R@10"],
                    f"{dataset_tag} ΔP@1": delta_p1
                }

            qd_de = eval_qd_pair(eu_de, "EU-DE")
            dd_de = eval_dd_embedder(model, eu_de)

            qd_fr = eval_qd_pair(eu_fr, "EU-FR")
            dd_fr = eval_dd_embedder(model, eu_fr)

            qd_mix = eval_qd_pair(eu_mix, "EU-MIX")
            dd_mix = eval_dd_embedder(model, eu_mix)

            # ---------------- Europeana (OCR bins) ----------------
            bin_rows = []
            # DE bins
            for i, (lo, hi) in enumerate(de_bins, 1):
                subset = filter_by_ocr_bin(eu_de, lo, hi)
                if len(subset) < MIN_BIN_SIZE:
                    continue
                metrics = eval_qd_pair(subset, f"EU-DE[mean_ocr∈{lo:.3f},{hi:.3f}]")
                metrics["SubsetSize"] = len(subset)
                metrics["Bin"] = f"DE_bin{i}"
                bin_rows.append(metrics)
            # FR bins
            for i, (lo, hi) in enumerate(fr_bins, 1):
                subset = filter_by_ocr_bin(eu_fr, lo, hi)
                if len(subset) < MIN_BIN_SIZE:
                    continue
                metrics = eval_qd_pair(subset, f"EU-FR[mean_ocr∈{lo:.3f},{hi:.3f}]")
                metrics["SubsetSize"] = len(subset)
                metrics["Bin"] = f"FR_bin{i}"
                bin_rows.append(metrics)

            # ---------------- Europeana cross-lingual (mixed pool, embedder only) ----------------
            xl = eval_europeana_xl_mixed(model, model_name, eu_fr, eu_de)

            # ---------------- AmericanStories (EN) ----------------
            as_qd = eval_qd_pair(amer, "AS")
            as_dd = eval_dd_embedder(model, amer)

            # ---------------- Kubhist2 (SV) ----------------
            kb_qd = eval_qd_pair(kub, "KUB")
            kb_dd = eval_dd_embedder(model, kub)

            row = {
                "ModelDir": mdir,

                # Europeana full
                **qd_de, "EU-DE P@1(dd)": dd_de["P@1"], "EU-DE R@5(dd)": dd_de["R@5"], "EU-DE R@10(dd)": dd_de["R@10"],
                **qd_fr, "EU-FR P@1(dd)": dd_fr["P@1"], "EU-FR R@5(dd)": dd_fr["R@5"], "EU-FR R@10(dd)": dd_fr["R@10"],
                **qd_mix, "EU-MIX P@1(dd)": dd_mix["P@1"], "EU-MIX R@5(dd)": dd_mix["R@5"], "EU-MIX R@10(dd)": dd_mix["R@10"],

                # XL mixed (embedder only)
                **xl,

                # Kubhist2
                **kb_qd, "KUB P@1(dd)": kb_dd["P@1"], "KUB R@5(dd)": kb_dd["R@5"], "KUB R@10(dd)": kb_dd["R@10"],
            }
            all_rows.append(row)

            # Save OCR-bin rows per model
            if bin_rows:
                bins_df = pd.DataFrame(bin_rows)
                bins_out = f"{OCR_DIR}/results/ocr_bins_{os.path.basename(mdir)}.csv"
                bins_df.to_csv(bins_out, index=False)
                print("Saved OCR-bin results:", bins_out)

    # Save main table
    if all_rows:
        df = pd.DataFrame(all_rows)
        out = f"{OCR_DIR}/results/eval_historic_ocr_xl_plus.csv"
        df.to_csv(out, index=False)
        print("Saved:", out)
    else:
        print("No results to save.")

MODEL_PREFIXES = [
    "Alibaba-NLP_gte-multilingual-base",
    "impresso-project_histlux-gte-multilingual-base",
    ]
run_all(MODEL_PREFIXES)


Using device: cuda


Resolving data files:   0%|          | 0/94 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/26 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/94 [00:00<?, ?it/s]

RuntimeError: Dataset scripts are no longer supported, but found AmericanStories.py